# CSMR-v1 FREEZE CHECK — paired seed-100 replication

Training seed **100** changes RNG only. Development Train/Val remain the frozen **seed42 Stage 0 manifest** (5,000/2,000). Train two independent ImageNet-initialized MobileNetV3-Small models, Vanilla then fixed CSMR-v1. Select checkpoints and thresholds only on micro-Val; evaluate LCC once after both are frozen. Write the pretest freeze decision to disk **before** reading official Test metadata, then open deterministic Test-2000 only on PASS. LCC is external development; Test is descriptive post-freeze confirmation. No Test-driven tuning or resampling.

After a persisted PASS, this notebook builds a Test-only bbox cache for exactly the selected 2,000 official Test keys using the verbatim E1 v5.3 SCRFD generator and the same detector model SHA/policy as Stage 0. The 7K Stage 0 micro bbox cache is insufficient. If InsightFace/ONNX Runtime or the exact detector model is missing, execution stops without changing crop policy.


In [18]:
# DATA ROOTS / KAGGLE CONFIG — change paths only for the attached datasets.
STAGE0_RESOURCE_DIR = (
    "/kaggle/input/datasets/maithanhphuong/"
    "face-auth-stage0-micro-v1"
)
STAGE1_RESULT_DIR = (
    "/kaggle/input/datasets/maithanhphuong/"
    "smartface-pad-micro-stage1-v1/"
    "stage1_shared_micro_e1"
)
CELEBA_ROOT = (
    "/kaggle/input/datasets/attentionlayer241/"
    "celeba-spoof-for-face-antispoofing/"
    "CelebA_Spoof_/CelebA_Spoof"
)
LCC_FASD_ROOT = (
    "/kaggle/input/datasets/aleksandrpikul222/"
    "lcc-fasd/"
    "LCC_FASD"
)
IMAGENET_WEIGHTS_PATH = ""
# Existing buffalo_s model root containing models/buffalo_s/det_500m.onnx.
# Attach the exact Stage-0 detector model if it is not present here.
SCRFD_MODEL_ROOT = "/root/.insightface"
OUTPUT_ROOT = "/kaggle/working/csmr_v1_freeze_check_seed100"

SEED = 100  # Training RNG only; Stage 0 still uses micro_manifest_v1_seed42.npz.
BATCH_SIZE = 128  # Only 64 is allowed for an OOM rerun.
NUM_WORKERS = 4
AMP = True
MAX_EPOCHS = 6
EARLY_STOP_PATIENCE = 2
EARLIEST_EARLY_STOP_EPOCH = 3
BACKBONE_LR = 1e-5
HEAD_LR = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
GRAD_CLIP = 5.0
CSMR_WARMUP_EPOCHS = 1
CSMR_LAMBDA = 0.20
CSMR_BAND_CENTERS = [0.15, 0.45, 0.75]
CSMR_BAND_SIGMA = 0.10
CSMR_GAIN_RANGE = [0.65, 0.90]
CSMR_PRESERVE_DC = True
SAVE_INTERVENTION_GALLERY = True
SELECTION_CHUNK_SIZE = 32
TEST_SUBSET_N = 2000
TEST_SUBSET_SEED = 4242


In [19]:
import csv
import hashlib
import json
import math
import os
import random
import shutil
import time
import zipfile
from collections import Counter
from pathlib import Path, PurePosixPath

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import MobileNet_V3_Small_Weights

CELEBA_MEAN = np.asarray([0.5931, 0.4690, 0.4229], dtype=np.float32)
CELEBA_STD = np.asarray([0.2471, 0.2214, 0.2157], dtype=np.float32)
INPUT_SIZE = 224
E1_ARCHITECTURE = "MobileNetV3-Small-GAP-576-256-BN-HSwish-Dropout02-128-3-v1"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_text(value):
    return hashlib.sha256(value.encode("utf-8")).hexdigest()

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def find_unique_file(name, root=None, explicit_dir=""):
    if not explicit_dir:
        raise RuntimeError(f"Set the explicit directory containing {name}; broad Kaggle search is disabled")
    base = Path(explicit_dir).expanduser()
    path = base if base.is_file() else base / name
    if path.name != name or not path.is_file():
        raise FileNotFoundError(f"Expected {name} at {path}")
    return path.resolve()


def find_stage0_dir():
    root = Path(STAGE0_RESOURCE_DIR).expanduser()
    if not STAGE0_RESOURCE_DIR or not root.is_dir():
        raise FileNotFoundError(f"Stage 0 directory missing: {root}")
    if (root / "stage0_summary.json").is_file():
        return root.resolve()
    if (root / "stage0" / "stage0_summary.json").is_file():
        return (root / "stage0").resolve()
    raise FileNotFoundError(f"stage0_summary.json missing directly under {root} or {root / 'stage0'}")


def find_celeba_root():
    root = Path(CELEBA_ROOT).expanduser()
    if not (root / "metas/intra_test/train_label.json").is_file():
        raise FileNotFoundError(f"CelebA root is wrong: {root}")
    return root.resolve()


def find_lcc_root(manifest_path):
    root = Path(LCC_FASD_ROOT).expanduser()
    if not root.is_dir():
        raise FileNotFoundError(f"LCC root is wrong: {root}")
    return root.resolve()

CELEBA_BBOX_STATUS_FACTORS = {
    "VALID": 1.55,
    "FALLBACK_DET_FAIL": 1.50,
    "FALLBACK_SUSPICIOUS": 1.50,
}


def validate_bbox_record(record, domain, key):
    if not isinstance(record, dict):
        raise TypeError("BBox record must be a mapping for {}".format(key))
    status = str(record.get("status", ""))
    if domain == "celeba":
        if status == "INVALID":
            raise RuntimeError("Selected micro key has INVALID E1 bbox record: {}".format(key))
        if status not in CELEBA_BBOX_STATUS_FACTORS:
            raise RuntimeError("Unknown/unusable E1 bbox status {} for {}".format(status, key))
        expected_factor = CELEBA_BBOX_STATUS_FACTORS[status]
    elif domain == "lcc":
        if status != "VALID":
            raise RuntimeError("Only Stage 0 LCC VALID records may be scored: {} ({})".format(key, status))
        expected_factor = 1.55
    else:
        raise ValueError("Unknown PAD crop domain: {}".format(domain))
    if "crop_factor" not in record:
        raise KeyError("BBox record is missing crop_factor for {}".format(key))
    try:
        crop_factor = float(record["crop_factor"])
    except (TypeError, ValueError) as exc:
        raise ValueError("Invalid crop_factor for {}: {!r}".format(key, record.get("crop_factor"))) from exc
    if not np.isfinite(crop_factor) or not np.isclose(crop_factor, expected_factor, rtol=0.0, atol=1e-6):
        raise ValueError(
            "BBox status/crop-factor mismatch for {}: status={} factor={} expected={}".format(
                key, status, crop_factor, expected_factor
            )
        )
    bbox = np.asarray(record.get("bbox_xyxy", []), dtype=np.float64)
    if bbox.shape != (4,) or not np.isfinite(bbox).all() or bbox[2] <= bbox[0] or bbox[3] <= bbox[1]:
        raise ValueError("Invalid bbox_xyxy for {}: {!r}".format(key, record.get("bbox_xyxy")))
    return crop_factor


def ordered_path_fingerprint(paths):
    return sha256_text("\n".join(map(str, paths)))


def assert_ordered_keys(actual, expected, name):
    actual = list(map(str, actual))
    expected = list(map(str, expected))
    if actual != expected:
        raise RuntimeError(
            "{} sample keys/order differ from the frozen manifest: actual_n={}, expected_n={}".format(
                name, len(actual), len(expected)
            )
        )


def labels_from_attack_codes(codes):
    codes = np.asarray(codes, dtype=np.int64)
    labels = np.full(codes.shape, 2, dtype=np.int64)
    labels[codes == 0] = 0
    labels[np.isin(codes, np.arange(1, 8))] = 1
    return labels

def load_stage0_resources():
    stage0 = find_stage0_dir()
    manifest_path = stage0 / "micro_manifest_v1_seed42.npz"
    bbox_path = stage0 / "celeba_micro_bbox_cache.json"
    lcc_manifest_path = stage0 / "lcc_external_dev_manifest.csv"
    lcc_cache_path = stage0 / "lcc_scrfd_cache_micro_v1.json"
    fingerprints_path = stage0 / "micro_resource_fingerprints.json"
    summary_path = stage0 / "stage0_summary.json"
    for path in [manifest_path, bbox_path, lcc_manifest_path, lcc_cache_path, fingerprints_path, summary_path]:
        if not path.is_file():
            raise FileNotFoundError("Stage 0 resource missing: {}".format(path))
    fingerprints = json.loads(fingerprints_path.read_text())
    stage0_summary = json.loads(summary_path.read_text())
    if stage0_summary.get("schema") != "pad_micro_stage0_summary_v1":
        raise RuntimeError("Unsupported Stage 0 summary schema")
    for path, key in [
        (manifest_path, "micro_manifest_sha256"),
        (lcc_manifest_path, "lcc_manifest_sha256"),
        (lcc_cache_path, "lcc_cache_sha256"),
        (bbox_path, "celeba_bbox_cache_sha256"),
    ]:
        if sha256_file(path) != fingerprints.get(key):
            raise RuntimeError("Stage 0 SHA256 mismatch for {}".format(key))
    with np.load(manifest_path, allow_pickle=False) as pack:
        train_keys = pack["micro_train_keys"].astype(str)
        val_keys = pack["micro_val_keys"].astype(str)
        train_labels = pack["micro_train_labels"].astype(np.int64)
        val_labels = pack["micro_val_labels"].astype(np.int64)
        train_codes = pack["micro_train_attack_codes"].astype(np.int64)
        val_codes = pack["micro_val_attack_codes"].astype(np.int64)
    if len(train_keys) != 5000 or len(val_keys) != 2000:
        raise RuntimeError("Stage 0 must freeze exactly 5,000 Train and 2,000 Val keys")
    if len(set(train_keys) & set(val_keys)):
        raise RuntimeError("Stage 0 Train/Val keys overlap")
    for name, labels, codes in [
        ("Train", train_labels, train_codes),
        ("Val", val_labels, val_codes),
    ]:
        if labels.shape != codes.shape or set(map(int, labels)) != {0, 1, 2}:
            raise RuntimeError("Stage 0 {} label/code arrays are malformed or miss a class".format(name))
        if not np.array_equal(labels, labels_from_attack_codes(codes)):
            raise RuntimeError("Stage 0 {} labels do not match E1 v5.3 attack-code mapping".format(name))
    bbox_payload = json.loads(bbox_path.read_text())
    bbox_cache = bbox_payload.get("records", {})
    selected_keys = set(map(str, train_keys)) | set(map(str, val_keys))
    if bbox_payload.get("schema") != "celeba_micro_bbox_cache_v1" or set(bbox_cache) != selected_keys:
        raise RuntimeError("Stage 0 CelebA bbox cache keys/schema differ from the frozen micro manifest")
    for key, record in bbox_cache.items():
        validate_bbox_record(record, "celeba", key)
    lcc_df = pd.read_csv(lcc_manifest_path)
    lcc_payload = json.loads(lcc_cache_path.read_text())
    lcc_records = lcc_payload.get("records", {})
    if lcc_payload.get("schema") != "pad_micro_lcc_scrfd_cache_v1":
        raise RuntimeError("Unsupported Stage 0 LCC cache schema")
    if lcc_df["path"].astype(str).duplicated().any():
        raise RuntimeError("Stage 0 LCC manifest contains duplicate paths")
    if not set(lcc_df["label"].astype(int)).issubset({0, 1}):
        raise RuntimeError("Stage 0 LCC labels must be exactly 0=Real or 1=Attack")
    record_by_path = {}
    for record in lcc_records.values():
        path = str(record.get("path", ""))
        record_by_path.setdefault(path, []).append(record)
    for row in lcc_df.itertuples(index=False):
        matches = record_by_path.get(str(row.path), [])
        if len(matches) != 1:
            raise RuntimeError("Stage 0 LCC cache join for {} has {} rows".format(row.path, len(matches)))
        record = matches[0]
        if str(record.get("status", "")) != str(row.status):
            raise RuntimeError("Stage 0 LCC manifest/cache status mismatch for {}".format(row.path))
        if int(record.get("label", -1)) != int(row.label):
            raise RuntimeError("Stage 0 LCC manifest/cache label mismatch for {}".format(row.path))
        if str(record.get("frame_id", "")) != str(row.record_id):
            raise RuntimeError("Stage 0 LCC manifest record_id mismatch for {}".format(row.path))
        if str(row.status) == "VALID":
            validate_bbox_record(record, "lcc", row.path)
    valid_paths = lcc_df.loc[lcc_df["status"].astype(str) == "VALID", "path"].astype(str).tolist()
    valid_labels = lcc_df.loc[lcc_df["status"].astype(str) == "VALID", "label"].astype(int).tolist()
    if not valid_paths or set(valid_labels) != {0, 1}:
        raise RuntimeError("Common LCC VALID set must be non-empty and contain Real plus Attack")
    if len(lcc_df) != int(fingerprints.get("lcc_candidate_n", -1)):
        raise RuntimeError("Stage 0 LCC candidate count mismatch")
    if len(valid_paths) != int(fingerprints.get("lcc_valid_face_n", -1)):
        raise RuntimeError("Stage 0 LCC valid-face count mismatch")
    valid_path_sha = ordered_path_fingerprint(valid_paths)
    if valid_path_sha != fingerprints.get("lcc_valid_path_fingerprint_sha256"):
        raise RuntimeError("Stage 0 ordered LCC VALID path fingerprint mismatch")
    if stage0_summary.get("lcc_valid_path_fingerprint_sha256") != valid_path_sha:
        raise RuntimeError("Stage 0 summary ordered LCC VALID path fingerprint mismatch")
    return {
        "stage0_dir": stage0,
        "train_keys": train_keys,
        "val_keys": val_keys,
        "train_labels": train_labels,
        "val_labels": val_labels,
        "train_codes": train_codes,
        "val_codes": val_codes,
        "bbox_cache": bbox_cache,
        "lcc_df": lcc_df,
        "lcc_cache": lcc_records,
        "lcc_valid_paths": valid_paths,
        "fingerprints": fingerprints,
    }
def assert_raw_image_paths(resources, celeba_root, lcc_root):
    celeba_keys = list(map(str, resources["train_keys"])) + list(map(str, resources["val_keys"]))
    lcc_paths = resources["lcc_df"]["path"].astype(str).tolist()
    missing_celeba = [str(Path(celeba_root) / key) for key in celeba_keys if not (Path(celeba_root) / key).is_file()]
    missing_lcc = [str(Path(lcc_root) / path) for path in lcc_paths if not (Path(lcc_root) / path).is_file()]
    if missing_celeba or missing_lcc:
        raise FileNotFoundError(
            "Raw PAD image preflight failed before training: CelebA missing={}, LCC missing={}".format(
                missing_celeba[:5], missing_lcc[:5]
            )
        )
    print("Verified raw image paths before training: CelebA selected={}, LCC candidates={}".format(
        len(celeba_keys), len(lcc_paths)
    ))


def runtime_crop_bgr(image, bbox_xyxy, factor):
    h, w = image.shape[:2]
    x1, y1, x2, y2 = map(float, bbox_xyxy)
    bw, bh = x2 - x1, y2 - y1
    if bw <= 0 or bh <= 0:
        raise ValueError(f"Invalid xyxy bbox: {bbox_xyxy}")
    extent = max(bw, bh) * float(factor)
    side = int(extent)
    cx, cy = x1 + bw / 2.0, y1 + bh / 2.0
    xs, ys = int(cx - extent / 2.0), int(cy - extent / 2.0)
    xa, ya, xb, yb = max(0, xs), max(0, ys), min(w, xs + side), min(h, ys + side)
    if xb <= xa or yb <= ya:
        raise RuntimeError("Empty crop intersection")
    crop = image[ya:yb, xa:xb]
    return cv2.copyMakeBorder(
        crop, max(0, -ys), max(0, ys + side - h),
        max(0, -xs), max(0, xs + side - w), cv2.BORDER_REFLECT_101
    )

def resize_letterbox_rgb(rgb, size=224):
    h, w = rgb.shape[:2]
    ratio = size / max(h, w)
    nh, nw = max(1, int(h * ratio)), max(1, int(w * ratio))
    interp = cv2.INTER_LANCZOS4 if ratio > 1 else cv2.INTER_AREA
    image = cv2.resize(rgb, (nw, nh), interpolation=interp)
    dh, dw = size - nh, size - nw
    return cv2.copyMakeBorder(image, dh // 2, dh - dh // 2, dw // 2, dw - dw // 2, cv2.BORDER_REFLECT_101)

def apply_e1_mild_augmentation(rgb):
    if random.random() < 0.5:
        rgb = cv2.flip(rgb, 1)
    if random.random() < 0.30:
        alpha = random.uniform(0.90, 1.10)
        beta = random.uniform(-0.10, 0.10) * 255.0
        rgb = np.clip(rgb.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
    return rgb

def bbox_jitter(record):
    x1, y1, x2, y2 = map(float, record["bbox_xyxy"])
    w, h = x2 - x1, y2 - y1
    if random.random() < 0.20:
        base = max(w, h)
        cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        scale = random.uniform(0.95, 1.05)
        cx += random.uniform(-0.05, 0.05) * base
        cy += random.uniform(-0.05, 0.05) * base
        w, h = w * scale, h * scale
        return [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]
    return [x1, y1, x2, y2]

def rgb_tensor(rgb):
    unit = rgb.astype(np.float32) / 255.0
    unit = (unit - CELEBA_MEAN) / CELEBA_STD
    return torch.from_numpy(np.ascontiguousarray(unit.transpose(2, 0, 1), dtype=np.float32))

def optional_crop_index(stage0_dir):
    path = Path(stage0_dir) / "crop_cache" / "index.json"
    return json.loads(path.read_text()) if path.is_file() else {}

def read_rgb_crop(image_path, record, domain, key, crop_index, stage0_dir, augment=False):
    crop_factor = validate_bbox_record(record, domain, key)
    cache_key = "{}::{}".format(domain, key)
    if not augment and cache_key in crop_index:
        path = Path(stage0_dir) / "crop_cache" / crop_index[cache_key]
        bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if bgr is None:
            raise FileNotFoundError("Optional Stage 0 crop cache entry is unreadable: {}".format(path))
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(image_path)
    box = bbox_jitter(record) if augment else record["bbox_xyxy"]
    crop = runtime_crop_bgr(bgr, box, crop_factor)
    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    if augment:
        rgb = apply_e1_mild_augmentation(rgb)
    return resize_letterbox_rgb(rgb, INPUT_SIZE)
class CelebAMicroDataset(Dataset):
    def __init__(self, keys, labels, codes, root, bbox_records, stage0_dir, train=False):
        self.keys = list(map(str, keys))
        self.labels = np.asarray(labels, dtype=np.int64)
        self.codes = np.asarray(codes, dtype=np.int64)
        self.root = Path(root)
        self.records = bbox_records
        self.stage0_dir = Path(stage0_dir)
        self.crop_index = optional_crop_index(stage0_dir)
        self.train = bool(train)
        if len(self.keys) != len(self.labels) or len(self.keys) != len(self.codes):
            raise ValueError("CelebA key/label/code lengths differ")
        missing = [k for k in self.keys if k not in self.records]
        if missing:
            raise RuntimeError(f"{len(missing)} CelebA keys lack Stage 0 bbox records")
    def __len__(self):
        return len(self.keys)
    def __getitem__(self, i):
        key = self.keys[i]
        rgb = read_rgb_crop(
            self.root / key, self.records[key], "celeba", key,
            self.crop_index, self.stage0_dir, augment=self.train
        )
        return rgb_tensor(rgb), int(self.labels[i]), key, int(self.codes[i])

class LCCValidDataset(Dataset):
    def __init__(self, frame, root, records, stage0_dir):
        self.frame = frame.reset_index(drop=True)
        self.root = Path(root)
        self.records = records
        self.stage0_dir = Path(stage0_dir)
        self.crop_index = optional_crop_index(stage0_dir)
        self.rows = []
        by_path = {}
        for rec in records.values():
            by_path.setdefault(str(rec.get("path", "")), []).append(rec)
        for _, row in self.frame.iterrows():
            if str(row["status"]) != "VALID":
                continue
            path = str(row["path"])
            candidates = by_path.get(path, [])
            if len(candidates) != 1:
                raise RuntimeError(f"LCC cache join for {path} has {len(candidates)} records")
            self.rows.append((path, int(row["label"]), candidates[0]))
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        key, label, record = self.rows[i]
        rgb = read_rgb_crop(
            self.root / key, record, "lcc", key, self.crop_index,
            self.stage0_dir, augment=False
        )
        return rgb_tensor(rgb), int(label), key, record.get("status", "VALID")

class SpatialE1(nn.Module):
    def __init__(self, imagenet_state=None):
        super().__init__()
        base = models.mobilenet_v3_small(weights=None)
        if imagenet_state is not None:
            missing, unexpected = base.load_state_dict(imagenet_state, strict=False)
            allowed_missing = [k for k in missing if k.startswith("classifier.")]
            bad_missing = [k for k in missing if k not in allowed_missing]
            allowed_unexpected = [k for k in unexpected if k.startswith("classifier.")]
            bad_unexpected = [k for k in unexpected if k not in allowed_unexpected]
            if bad_missing or bad_unexpected:
                raise RuntimeError(
                    "ImageNet MobileNetV3-Small state_dict is incomplete/incompatible; "
                    "backbone missing={}, unexpected={}".format(bad_missing[:5], bad_unexpected[:5])
                )
        self.features = base.features
        self.avgpool = base.avgpool
        self.projection = nn.Sequential(
            nn.Linear(576, 256),
            nn.BatchNorm1d(256),
            nn.Hardswish(),
            nn.Dropout(0.2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.Hardswish(),
            nn.Dropout(0.2),
            nn.Linear(128, 3),
        )
    def forward_features(self, x):
        x = self.features(x)
        x = self.avgpool(x).flatten(1)
        return self.projection(x)
    def forward(self, x):
        feature = self.forward_features(x)
        return self.classifier(feature)

def load_imagenet_weights(local_path=""):
    if local_path:
        state = torch.load(local_path, map_location="cpu", weights_only=False)
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        if not isinstance(state, dict):
            raise RuntimeError("Local ImageNet weights must be a state_dict")
        return state
    try:
        return MobileNet_V3_Small_Weights.DEFAULT.get_state_dict(progress=True)
    except Exception as exc:
        raise RuntimeError(
            "ImageNet MobileNetV3-Small weights are unavailable. Enable Kaggle Internet "
            "or set IMAGENET_WEIGHTS_PATH to a compatible torchvision state_dict. "
            "Random initialization is intentionally not used."
        ) from exc

def binary_score(logits):
    logits = torch.as_tensor(logits)
    if logits.ndim != 2 or logits.shape[1] != 3:
        raise ValueError("PAD logits must have shape [N, 3] for Real/Physical/Digital")
    if not torch.isfinite(logits).all():
        raise ValueError("PAD logits contain NaN or Infinity")
    return logits[:, 0] - torch.logsumexp(logits[:, 1:], dim=1)
def safe_auc(y_real, scores):
    from sklearn.metrics import roc_auc_score
    y_real = np.asarray(y_real, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y_real.ndim != 1 or scores.ndim != 1 or len(y_real) != len(scores) or len(y_real) == 0:
        raise ValueError("AUC needs non-empty one-dimensional labels and scores of equal length")
    if not np.isfinite(scores).all():
        raise ValueError("AUC scores contain NaN or Infinity")
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("AUC requires both binary classes (positive class is Real)")
    return float(roc_auc_score(y_real, scores))
def threshold_from_val(y3, scores):
    y3 = np.asarray(y3, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y3.ndim != 1 or scores.ndim != 1 or len(y3) != len(scores) or len(y3) == 0:
        raise ValueError("Threshold calibration needs non-empty 1-D labels/scores with equal lengths")
    if not set(map(int, np.unique(y3))).issubset({0, 1, 2}):
        raise ValueError("CelebA labels must be 0=Real, 1=Physical, 2=Digital")
    y_real = (y3 == 0).astype(np.int64)
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("Threshold calibration requires at least one Real and one Attack sample")
    if not np.isfinite(scores).all():
        raise ValueError("Threshold calibration scores contain NaN or Infinity")
    values = np.unique(scores)
    lower = np.nextafter(values[0], -np.inf)
    upper = np.nextafter(values[-1], np.inf)
    if not np.isfinite(lower):
        lower = values[0]
    if not np.isfinite(upper):
        upper = values[-1]
    midpoints = values[:-1] / 2.0 + values[1:] / 2.0
    candidates = np.concatenate(([lower], midpoints, [upper]))
    best = None
    for threshold in candidates:
        pred_real = scores >= threshold
        real = y_real == 1
        attack = y_real == 0
        bpcer = float(np.mean(~pred_real[real]))
        apcer = float(np.mean(pred_real[attack]))
        acer = 0.5 * (apcer + bpcer)
        key = (acer, abs(float(threshold)), float(threshold))
        if best is None or key < best[0]:
            best = (key, float(threshold))
    if best is None or not np.isfinite(best[1]):
        raise RuntimeError("Threshold search did not produce a finite threshold")
    return best[1]
def metrics_binary(y3, scores, threshold, coverage=None, candidate_n=None):
    y3 = np.asarray(y3, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y3.ndim != 1 or scores.ndim != 1 or len(y3) != len(scores) or len(y3) == 0:
        raise ValueError("PAD metrics need non-empty 1-D labels/scores with equal lengths")
    if not set(map(int, np.unique(y3))).issubset({0, 1, 2}):
        raise ValueError("PAD labels must be from the frozen class mapping 0/1/2")
    if not np.isfinite(scores).all() or not np.isfinite(float(threshold)):
        raise ValueError("PAD scores and threshold must be finite")
    y_real = (y3 == 0).astype(np.int64)
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("PAD metrics require at least one Real and one Attack sample")
    if coverage is not None and (not np.isfinite(float(coverage)) or not 0.0 <= float(coverage) <= 1.0):
        raise ValueError("Detector coverage must be finite and within [0, 1]")
    scored_n = len(y3)
    candidate_n = int(candidate_n if candidate_n is not None else scored_n)
    if candidate_n < scored_n:
        raise ValueError("candidate_n cannot be less than n_scored")
    pred_real = scores >= float(threshold)
    real = y_real == 1
    attack = y_real == 0
    bpcer = float(np.mean(~pred_real[real]))
    apcer = float(np.mean(pred_real[attack]))
    acer = 0.5 * (apcer + bpcer)
    return {
        "n_scored": int(scored_n),
        "candidate_n": candidate_n,
        "detector_coverage": float(coverage if coverage is not None else 1.0),
        "apcer": apcer,
        "bpcer": bpcer,
        "acer": acer,
        "hter": acer,
        "auc": safe_auc(y_real, scores),
        "accuracy": float(np.mean(pred_real == y_real)),
        "locked_logit_threshold": float(threshold),
    }
def metrics_celeba(y3, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    result = metrics_binary(y3, scores, threshold, coverage=1.0, candidate_n=len(y3))
    result["accuracy_3class"] = float(np.mean(np.argmax(logits, axis=1) == np.asarray(y3)))
    return result, scores

def make_prediction_frame(keys, y3, codes, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    pred_real = scores >= threshold
    result = pd.DataFrame({
        "key": list(map(str, keys)),
        "true_class": np.asarray(y3, dtype=np.int64),
        "attack_code": np.asarray(codes, dtype=np.int64),
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
        "pred_real": pred_real,
        "correct_binary": pred_real == (np.asarray(y3) == 0),
        "pred_class_3": np.argmax(logits, axis=1),
        "correct_3class": np.argmax(logits, axis=1) == np.asarray(y3),
    })
    return result

def eval_spatial(model, loader, device, amp):
    model.eval()
    logits_all, y_all, keys_all, codes_all = [], [], [], []
    with torch.no_grad():
        for x, y, keys, codes in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
            codes_all.extend(np.asarray(codes, dtype=np.int64).tolist())
    if not logits_all:
        raise RuntimeError("Spatial evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    labels = np.asarray(y_all, dtype=np.int64)
    codes = np.asarray(codes_all, dtype=np.int64)
    if len(logits) != len(labels) or len(keys_all) != len(labels) or len(codes) != len(labels):
        raise RuntimeError("Spatial evaluation outputs have inconsistent sample counts")
    return logits, labels, list(map(str, keys_all)), codes
def eval_lcc_spatial(model, loader, device, amp, lcc_frame, threshold):
    model.eval()
    logits_all, y_all, keys_all = [], [], []
    with torch.no_grad():
        for x, y, keys, _status in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
    if not logits_all:
        raise RuntimeError("Common LCC VALID evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    y = np.asarray(y_all, dtype=np.int64)
    keys = list(map(str, keys_all))
    expected_paths = lcc_frame.loc[lcc_frame["status"].astype(str) == "VALID", "path"].astype(str).tolist()
    assert_ordered_keys(keys, expected_paths, "Stage 1 LCC")
    if len(logits) != len(y) or len(keys) != len(y):
        raise RuntimeError("Stage 1 LCC logits/labels/keys have inconsistent counts")
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    scored = pd.DataFrame({
        "path": keys, "label": y,
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
    })
    full = lcc_frame[["path", "label", "status"]].merge(
        scored, on=["path", "label"], how="left", validate="one_to_one"
    )
    full["dataset_role"] = "LCC-FASD external-dev / cross-domain stress set"
    valid = full["status"].astype(str).eq("VALID")
    if full.loc[valid, "pad_logit_score"].isna().any() or int(valid.sum()) != len(keys):
        raise RuntimeError("Stage 1 inference failed to score the exact common LCC VALID set")
    full["pred_real"] = np.nan
    full.loc[valid, "pred_real"] = full.loc[valid, "pad_logit_score"] >= float(threshold)
    full["pred_real"] = full["pred_real"].where(valid, np.nan)
    return logits, y, keys, full
def trainable_parameter_count(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))

def write_zip(folder, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(Path(folder).rglob("*")):
            if path.is_file() and path.resolve() != Path(zip_path).resolve():
                archive.write(path, arcname=Path(Path(folder).name) / path.relative_to(folder))


In [20]:
import re
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BANDS = ('LOW', 'MID', 'HIGH')

class CSMRTrainRGBDataset(CelebAMicroDataset):
    def __init__(self, *args):
        super().__init__(*args, train=True)
    def __getitem__(self, i):
        key = self.keys[i]
        rgb = read_rgb_crop(self.root / key, self.records[key], 'celeba', key,
                            self.crop_index, self.stage0_dir, augment=True)
        x = torch.from_numpy(np.ascontiguousarray(rgb.transpose(2, 0, 1))).float() / 255.0
        return x, int(self.labels[i]), key, int(self.codes[i])

def normalize_unit(x):
    mean = torch.as_tensor(CELEBA_MEAN, device=x.device, dtype=x.dtype)[None, :, None, None]
    std = torch.as_tensor(CELEBA_STD, device=x.device, dtype=x.dtype)[None, :, None, None]
    return (x - mean) / std

def radial_masks(size=INPUT_SIZE, device='cpu'):
    y = torch.arange(size, device=device, dtype=torch.float32) - size // 2
    yy, xx = torch.meshgrid(y, y, indexing='ij')
    radius = torch.sqrt(xx.square() + yy.square()) / math.sqrt(2 * (size // 2) ** 2)
    centers = torch.as_tensor(CSMR_BAND_CENTERS, dtype=torch.float32, device=device)
    masks = torch.exp(-((radius[None] - centers[:, None, None]) ** 2) /
                      (2 * CSMR_BAND_SIGMA ** 2))
    if masks.shape != (3, size, size) or not torch.isfinite(masks).all() or not ((masks >= 0) & (masks <= 1)).all():
        raise RuntimeError('Invalid CSMR radial masks')
    if len(set(map(float, CSMR_BAND_CENTERS))) != 3:
        raise RuntimeError('CSMR band centers must differ')
    return masks, radius

def spectral_views(x, gains, masks):
    if x.ndim != 4 or x.shape[1:] != (3, INPUT_SIZE, INPUT_SIZE) or not torch.isfinite(x).all():
        raise ValueError('CSMR input must be finite [B,3,224,224]')
    x = x.float()
    gains = gains.to(device=x.device, dtype=torch.float32).reshape(-1)
    if len(gains) != len(x) or not ((gains >= 0) & (gains <= 1)).all():
        raise ValueError('Invalid gains')
    spectrum = torch.fft.fftshift(torch.fft.fft2(x, norm='ortho'), dim=(-2, -1))
    gain_fields = 1 - (1 - gains[:, None, None, None]) * masks[None]
    if CSMR_PRESERVE_DC:
        gain_fields[:, :, INPUT_SIZE // 2, INPUT_SIZE // 2] = 1.0
    perturbed = spectrum[:, None] * gain_fields[:, :, None]
    raw = torch.fft.ifft2(torch.fft.ifftshift(perturbed, dim=(-2, -1)), norm='ortho').real
    clipped = ((raw < 0) | (raw > 1)).float().mean(dim=(2, 3, 4))
    views = raw.clamp(0, 1)
    if not torch.isfinite(views).all():
        raise RuntimeError('Non-finite CSMR intervention')
    return views, gain_fields, clipped

def aligned_margin(logits, labels):
    d = binary_score(logits.float())
    labels = torch.as_tensor(labels, device=d.device)
    if not torch.isin(labels, torch.tensor([0, 1, 2], device=d.device)).all():
        raise ValueError('Invalid PAD label')
    return torch.where(labels == 0, d, -d)

def harmful_margin_loss(clean_margin, worst_margin):
    drop = clean_margin.detach() - worst_margin
    harmful = drop > 0
    per = F.smooth_l1_loss(worst_margin, clean_margin.detach(), reduction='none')
    return (per * harmful.float()).mean(), harmful, drop

def selection_by_margins(margins):
    if margins.ndim != 2 or margins.shape[1] != 3 or not torch.isfinite(margins).all():
        raise ValueError('Expected finite [B,3] margins')
    return margins.argmin(dim=1)

def select_worst_view(model, base, labels, masks, amp, chunk_size):
    if chunk_size < 1:
        raise ValueError('Selection chunk size must be positive')
    was_training = model.training
    model.eval()
    chosen, indexes, all_gains, all_clips = [], [], [], []
    try:
        with torch.no_grad():
            for start in range(0, len(base), chunk_size):
                x = base[start:start+chunk_size]
                y = labels[start:start+chunk_size]
                gains = torch.empty(len(x), device=x.device).uniform_(*CSMR_GAIN_RANGE)
                views, _, clips = spectral_views(x, gains, masks)
                margins = []
                for band in range(3):
                    with torch.cuda.amp.autocast(enabled=amp):
                        logits = model(normalize_unit(views[:, band]))
                    margins.append(aligned_margin(logits, y))
                indexes_chunk = selection_by_margins(torch.stack(margins, dim=1))
                ix = torch.arange(len(x), device=x.device)
                chosen.append(views[ix, indexes_chunk].detach())
                indexes.append(indexes_chunk.detach())
                all_gains.append(gains.detach())
                all_clips.append(clips[ix, indexes_chunk].detach())
    finally:
        model.train(was_training)
    return tuple(torch.cat(a) for a in (chosen, indexes, all_gains, all_clips))

def strict_json_load(path):
    def reject(value):
        raise ValueError('Non-finite JSON constant: ' + value)
    return json.loads(Path(path).read_text(), parse_constant=reject)

def strict_json_write(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2, allow_nan=False) + '\n')

def load_stage1_reference(resources):
    path = find_unique_file('micro_e1_v1_summary.json', explicit_dir=STAGE1_RESULT_DIR)
    summary = strict_json_load(path)
    if summary.get('schema') != 'pad_micro_stage1_summary_v1':
        raise RuntimeError('Stage 1 baseline summary schema mismatch')
    if summary.get('resource_hashes') != resources['fingerprints']:
        raise RuntimeError('Stage 0/Stage 1 resource hash mismatch')
    digest = summary.get('checkpoint_sha256', '')
    if not isinstance(digest, str) or not re.fullmatch(r'[0-9a-f]{64}', digest):
        raise RuntimeError('Invalid Stage 1 checkpoint SHA256')
    for name in ('micro_e1_v1_meta.json', 'micro_e1_v1_config.json'):
        candidate = Path(path).parent / name
        if candidate.exists():
            companion = strict_json_load(candidate)
            if companion.get('checkpoint_sha256') != digest or companion.get('resource_hashes') != resources['fingerprints']:
                raise RuntimeError('Stage 1 companion SHA/resource mismatch: ' + name)
    candidate = Path(path).parent / 'micro_e1_v1_best.pth'
    if candidate.exists() and sha256_file(candidate) != digest:
        raise RuntimeError('Stage 1 checkpoint file SHA mismatch')
    for split, n in (('val', 2000), ('lcc', len(resources['lcc_valid_paths']))):
        m = summary.get(split, {})
        if int(m.get('n_scored', -1)) != n:
            raise RuntimeError('Stage 1 ' + split + ' count mismatch')
        for metric in ('acer', 'auc', 'apcer', 'bpcer', 'hter'):
            if not math.isfinite(float(m[metric])):
                raise RuntimeError('Non-finite Stage 1 metric')
        if float(m['locked_logit_threshold']) != float(summary['locked_validation_logit_threshold']):
            raise RuntimeError('Stage 1 threshold mismatch')
    return summary, path

def intervention_sanity(masks, device):
    gen = torch.Generator(device=device).manual_seed(SEED)
    x = torch.rand((2, 3, INPUT_SIZE, INPUT_SIZE), generator=gen, device=device)
    gain = torch.tensor([0.65, 0.90], device=device)
    views, fields, clips = spectral_views(x, gain, masks)
    assert views.shape == (2, 3, 3, INPUT_SIZE, INPUT_SIZE)
    assert torch.all(fields[:, :, INPUT_SIZE//2, INPUT_SIZE//2] == 1)
    unity, _, _ = spectral_views(x, torch.ones(2, device=device), masks)
    assert torch.max(torch.abs(unity[:, 0] - x)).item() < 1e-5
    ft = torch.fft.fftshift(torch.fft.fft2(x, norm='ortho'), dim=(-2,-1))
    modified = ft[:, None] * fields[:, :, None]
    valid = ft.abs()[:, None] > 1e-5
    phase_error = (torch.angle(modified) - torch.angle(ft[:, None])).abs()
    phase_error = torch.minimum(phase_error, 2*math.pi-phase_error)
    assert phase_error.expand_as(valid.expand_as(modified))[valid.expand_as(modified)].max().item() < 1e-4
    report = {}
    for i, band in enumerate(BANDS):
        diff = (views[:, i] - x).abs()
        report[band] = {'mean_absolute_pixel_difference': float(diff.mean()),
                        'max_absolute_pixel_difference': float(diff.max()),
                        'clipped_pixel_fraction': float(clips[:, i].mean())}
        if report[band]['clipped_pixel_fraction'] > 0.05:
            print('WARNING: clipping exceeds 5% for', band)
    return report

def save_gallery(train_ds, masks, out_dir):
    fig, ax = plt.subplots(1, 4, figsize=(16, 4))
    radius = radial_masks(device='cpu')[1].numpy()
    ax[0].imshow(radius, cmap='viridis', vmin=0, vmax=1); ax[0].set_title('r: center=0, corner≈1')
    for i, band in enumerate(BANDS):
        ax[i+1].imshow(masks[i].detach().cpu().numpy(), cmap='viridis', vmin=0, vmax=1)
        ax[i+1].set_title(f'{band} c={CSMR_BAND_CENTERS[i]}')
    for a in ax: a.axis('off')
    fig.tight_layout(); fig.savefig(out_dir/'csmr_v1_frequency_masks.png', dpi=130); plt.close(fig)
    rng = np.random.default_rng(SEED)
    indices = []
    for cls in range(3):
        pool = np.where(train_ds.labels == cls)[0]
        if len(pool) < 2: raise RuntimeError('Gallery requires 2 examples per class')
        indices.extend(rng.choice(pool, 2, replace=False).tolist())
    fig, ax = plt.subplots(6, 4, figsize=(16, 20))
    for row, idx in enumerate(indices):
        random.seed(SEED + idx); np.random.seed(SEED + idx)
        x, label, key, _ = train_ds[idx]
        gain = float(rng.uniform(*CSMR_GAIN_RANGE))
        views, _, _ = spectral_views(x[None], torch.tensor([gain]), masks.cpu())
        for col, item in enumerate([x, *list(views[0])]):
            ax[row,col].imshow(item.permute(1,2,0).numpy())
            ax[row,col].set_title(f'{["Clean", *BANDS][col]} | class={label} | g={gain:.3f}\n{key[-45:]}')
            ax[row,col].axis('off')
    fig.tight_layout(); fig.savefig(out_dir/'csmr_v1_intervention_gallery.png', dpi=110); plt.close(fig)


In [21]:
from datetime import datetime, timezone
from sklearn.model_selection import train_test_split
from IPython.display import FileLink, display

RUN_STARTED = time.monotonic()

def progress(section, message):
    stamp = datetime.now(timezone.utc).isoformat(timespec='seconds')
    print(f'[{stamp} | +{time.monotonic()-RUN_STARTED:,.1f}s | {section}] {message}', flush=True)

def gpu_usage():
    if not torch.cuda.is_available():
        return 'CUDA unavailable'
    return (f'GPU {torch.cuda.get_device_name(0)} | allocated '
            f'{torch.cuda.memory_allocated()/2**20:.0f} MiB | '
            f'reserved {torch.cuda.memory_reserved()/2**20:.0f} MiB | '
            f'peak {torch.cuda.max_memory_allocated()/2**20:.0f} MiB')

def check_fixed_contract():
    if (SEED != 100 or BATCH_SIZE not in (128, 64) or MAX_EPOCHS != 6 or
        EARLY_STOP_PATIENCE != 2 or EARLIEST_EARLY_STOP_EPOCH != 3 or
        BACKBONE_LR != 1e-5 or HEAD_LR != 1e-4 or WEIGHT_DECAY != 1e-4 or
        LABEL_SMOOTHING != .1 or GRAD_CLIP != 5.0 or
        CSMR_WARMUP_EPOCHS != 1 or CSMR_LAMBDA != .2 or
        CSMR_BAND_CENTERS != [.15,.45,.75] or CSMR_BAND_SIGMA != .1 or
        CSMR_GAIN_RANGE != [.65,.90] or CSMR_PRESERVE_DC is not True or
        SAVE_INTERVENTION_GALLERY is not True or SELECTION_CHUNK_SIZE < 1 or
        TEST_SUBSET_N != 2000 or TEST_SUBSET_SEED != 4242):
        raise RuntimeError('Fixed seed-100 freeze-check contract changed; only BATCH_SIZE=64 is an OOM rerun option')

def make_optimizer(model):
    return torch.optim.AdamW([
        {'params': list(model.features.parameters())+list(model.avgpool.parameters()), 'lr':BACKBONE_LR},
        {'params': list(model.projection.parameters())+list(model.classifier.parameters()), 'lr':HEAD_LR},
    ], weight_decay=WEIGHT_DECAY)

def model_state_cpu(model):
    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

def evaluate_val(model, loader, resources, device, use_amp):
    logits,y,keys,codes = eval_spatial(model,loader,device,use_amp)
    assert_ordered_keys(keys,resources['val_keys'],'frozen micro-Val')
    threshold = threshold_from_val(y,binary_score(torch.as_tensor(logits)).cpu().numpy())
    metrics,_ = metrics_celeba(y,logits,threshold)
    return metrics,threshold,(logits,y,keys,codes)

def train_one(name, csmr, resources, celeba_root, imagenet_state, val_loader, device, use_amp, masks, output_root):
    progress(name,'Seeding training RNG=100 before model and DataLoader construction')
    seed_everything(SEED)
    train_ds = CSMRTrainRGBDataset(resources['train_keys'],resources['train_labels'],
        resources['train_codes'],celeba_root,resources['bbox_cache'],resources['stage0_dir'])
    out_dir=output_root/name
    out_dir.mkdir()
    if csmr:
        sanity=intervention_sanity(masks,device)
        progress(name,f'FFT sanity: {sanity}')
        save_gallery(train_ds,masks,out_dir)
        progress(name,'Saved mask figure and 6-sample intervention gallery')
        seed_everything(SEED)  # Same model/loader initialization despite gallery RNG.
    else:
        sanity=None
    model=SpatialE1(imagenet_state).to(device)
    progress(name,f'Independent ImageNet model; params={trainable_parameter_count(model)}; {gpu_usage()}')
    loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,drop_last=False,
        num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=NUM_WORKERS>0)
    progress(name,f'Train batches={len(loader)}; logging every {max(1,len(loader)//18)} batches')
    optimizer=make_optimizer(model)
    criterion=nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
    history=[]; diagnostics=[]; class_rows=[]
    best_acer=float('inf'); best_epoch=None; best_state=None
    try:
        for epoch in range(1,MAX_EPOCHS+1):
            epoch_started=time.monotonic();model.train()
            progress(name,f'Epoch {epoch}/{MAX_EPOCHS} start; {gpu_usage()}')
            totals=[]; cls_losses=[]; margin_losses=[]
            band_counts=torch.zeros(3,dtype=torch.int64)
            class_counts=torch.zeros((3,3),dtype=torch.int64)
            class_harmful=torch.zeros(3,dtype=torch.int64)
            class_drop=torch.zeros(3,dtype=torch.float64)
            class_seen=torch.zeros(3,dtype=torch.int64)
            harmful_n=0; harmful_drop_sum=clean_margin_sum=worst_margin_sum=gain_sum=clip_sum=0.
            csmr_n=0
            for batch_idx,(base,labels,_keys,_codes) in enumerate(loader,1):
                base=base.to(device,non_blocking=True)
                labels=labels.to(device,non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                if csmr and epoch>CSMR_WARMUP_EPOCHS:
                    selected,band_idx,gains,clips=select_worst_view(
                        model,base,labels,masks,use_amp,SELECTION_CHUNK_SIZE)
                    model.train()
                    combined=torch.cat([normalize_unit(base),normalize_unit(selected)],dim=0)
                    with torch.cuda.amp.autocast(enabled=use_amp):
                        logits=model(combined)
                        clean_logits,worst_logits=logits.chunk(2)
                        cls_loss=.5*criterion(clean_logits,labels)+.5*criterion(worst_logits,labels)
                        clean_margin=aligned_margin(clean_logits,labels)
                        worst_margin=aligned_margin(worst_logits,labels)
                        margin_loss,harmful,drop=harmful_margin_loss(clean_margin,worst_margin)
                        loss=cls_loss+CSMR_LAMBDA*margin_loss
                    n=len(labels);csmr_n+=n
                    gain_sum+=float(gains.sum().cpu());clip_sum+=float(clips.sum().cpu())
                    harmful_n+=int(harmful.sum().cpu())
                    harmful_drop_sum+=float(drop.clamp_min(0).sum().detach().cpu())
                    clean_margin_sum+=float(clean_margin.sum().detach().cpu())
                    worst_margin_sum+=float(worst_margin.sum().detach().cpu())
                    for band in range(3):
                        band_counts[band]+=int((band_idx==band).sum().cpu())
                    for cls in range(3):
                        mask=labels==cls
                        class_seen[cls]+=int(mask.sum().cpu())
                        class_harmful[cls]+=int((harmful&mask).sum().cpu())
                        class_drop[cls]+=float(drop.clamp_min(0)[mask].sum().detach().cpu())
                        for band in range(3):
                            class_counts[cls,band]+=int((mask&(band_idx==band)).sum().cpu())
                else:
                    with torch.cuda.amp.autocast(enabled=use_amp):
                        clean_logits=model(normalize_unit(base))
                        cls_loss=criterion(clean_logits,labels)
                        margin_loss=cls_loss.detach()*0
                        loss=cls_loss
                if not torch.isfinite(loss):
                    raise RuntimeError(f'{name} epoch {epoch} batch {batch_idx}: non-finite loss')
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
                scaler.step(optimizer);scaler.update()
                totals.append(float(loss.detach().cpu()))
                cls_losses.append(float(cls_loss.detach().cpu()))
                margin_losses.append(float(margin_loss.detach().cpu()))
                if batch_idx==1 or batch_idx%max(1,len(loader)//18)==0 or batch_idx==len(loader):
                    progress(name,f'E{epoch} batch {batch_idx}/{len(loader)} loss={totals[-1]:.4f} mean={np.mean(totals):.4f} {gpu_usage()}')
            val_metrics,threshold,_=evaluate_val(model,val_loader,resources,device,use_amp)
            row={'epoch':epoch,'train_loss':float(np.mean(totals)),
                 'val_acer':val_metrics['acer'],'val_auc':val_metrics['auc'],
                 'val_apcer':val_metrics['apcer'],'val_bpcer':val_metrics['bpcer'],
                 'val_threshold':float(threshold),'seconds':time.monotonic()-epoch_started}
            history.append(row)
            progress(name,f'Epoch {epoch} end: {row}')
            if csmr:
                d={'epoch':epoch,'lambda':CSMR_LAMBDA if csmr_n else 0.,
                   'train_total_loss':float(np.mean(totals)),
                   'train_cls_loss':float(np.mean(cls_losses)),
                   'train_margin_loss':float(np.mean(margin_losses)) if csmr_n else None}
                for band,count in zip(BANDS,band_counts.tolist()):
                    d[f'worst_{band.lower()}_count']=count if csmr_n else None
                    d[f'worst_{band.lower()}_fraction']=count/csmr_n if csmr_n else None
                d.update({'harmful_fraction':harmful_n/csmr_n if csmr_n else None,
                    'mean_harmful_margin_drop':harmful_drop_sum/harmful_n if harmful_n else None,
                    'mean_clean_aligned_margin':clean_margin_sum/csmr_n if csmr_n else None,
                    'mean_worst_aligned_margin':worst_margin_sum/csmr_n if csmr_n else None,
                    'mean_sampled_gain':gain_sum/csmr_n if csmr_n else None,
                    'mean_clip_fraction':clip_sum/csmr_n if csmr_n else None})
                diagnostics.append(d)
                if csmr_n:
                    if max(d[f'worst_{b.lower()}_fraction'] for b in BANDS)>.95:
                        progress(name,'WARNING: one band selected for >95% of samples')
                    if d['mean_clip_fraction']>.05:
                        progress(name,'WARNING: mean clipped pixel fraction >5%')
                for cls,cls_name in enumerate(('Real','Physical','Digital')):
                    n=int(class_seen[cls])
                    cr={'epoch':epoch,'class':cls_name,'n':n if csmr_n else None,
                        'harmful_fraction':int(class_harmful[cls])/n if n else None,
                        'mean_margin_drop':float(class_drop[cls])/n if n else None}
                    for band in range(3):
                        cr[f'worst_{BANDS[band].lower()}_count']=int(class_counts[cls,band]) if n else None
                        cr[f'worst_{BANDS[band].lower()}_fraction']=int(class_counts[cls,band])/n if n else None
                    class_rows.append(cr)
            if val_metrics['acer']<best_acer:
                best_acer=val_metrics['acer'];best_epoch=epoch;best_state=model_state_cpu(model)
                progress(name,f'NEW BEST epoch={epoch} Val ACER={best_acer:.6f}')
            elif epoch>=EARLIEST_EARLY_STOP_EPOCH and epoch-best_epoch>=EARLY_STOP_PATIENCE:
                progress(name,'Early stop on frozen micro-Val ACER');break
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError('CUDA OOM: restart clean and change only BATCH_SIZE to 64') from exc
    if best_state is None:raise RuntimeError(f'{name}: no checkpoint selected')
    last_state=model_state_cpu(model)
    model.load_state_dict(best_state)
    final_val,locked,outputs=evaluate_val(model,val_loader,resources,device,use_amp)
    logits,y,keys,codes=outputs
    best_path=out_dir/'best.pth';last_path=out_dir/'checkpoint_last.pth'
    torch.save({'state_dict':best_state,'epoch':best_epoch,'val_acer':best_acer},best_path)
    torch.save({'state_dict':last_state,'epoch':epoch},last_path)
    checkpoint_sha=sha256_file(best_path)
    pd.DataFrame(history).to_csv(out_dir/'training_history.csv',index=False)
    if csmr:
        pd.DataFrame(diagnostics).to_csv(out_dir/'training_diagnostics.csv',index=False)
        pd.DataFrame(class_rows).to_csv(out_dir/'class_diagnostics.csv',index=False)
    make_prediction_frame(keys,y,codes,logits,locked).to_csv(out_dir/'val_predictions.csv',index=False)
    config={'schema':'csmr_seed100_experiment_config_v1','name':name,'training_seed':SEED,
        'development_manifest_name':'micro_manifest_v1_seed42.npz',
        'resource_hashes':resources['fingerprints'],'architecture':E1_ARCHITECTURE,
        'preprocessing':'pilot Stage-1 read_rgb_crop + normalize_unit; gamma off',
        'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,'patience':EARLY_STOP_PATIENCE,
        'backbone_lr':BACKBONE_LR,'head_lr':HEAD_LR,'weight_decay':WEIGHT_DECAY,
        'label_smoothing':LABEL_SMOOTHING,'grad_clip':GRAD_CLIP,'amp':use_amp,
        'csmr':csmr,'csmr_settings':({'warmup_epochs':CSMR_WARMUP_EPOCHS,
            'lambda':CSMR_LAMBDA,'centers':CSMR_BAND_CENTERS,'sigma':CSMR_BAND_SIGMA,
            'gain_range':CSMR_GAIN_RANGE,'preserve_dc':CSMR_PRESERVE_DC} if csmr else None)}
    strict_json_write(out_dir/'config.json',config)
    meta={'schema':'csmr_seed100_experiment_meta_v1','name':name,
        'checkpoint_file':best_path.name,'checkpoint_sha256':checkpoint_sha,
        'selected_epoch':int(best_epoch),'locked_val_threshold':float(locked),
        'threshold_source':'frozen CelebA micro-Val only',
        'val':final_val,'resource_hashes':resources['fingerprints'],
        'parameter_count':int(sum(p.numel() for p in model.parameters())),
        'stage1_checkpoint_loaded':False}
    strict_json_write(out_dir/'meta.json',meta)
    model.cpu()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    progress(name,f'Frozen best epoch={best_epoch}, Val ACER={final_val["acer"]:.6f}, threshold={locked:.6f}, SHA={checkpoint_sha}; {gpu_usage()}')
    return {'name':name,'model':model,'out_dir':out_dir,'val':final_val,
        'locked_threshold':float(locked),'selected_epoch':int(best_epoch),
        'checkpoint_sha256':checkpoint_sha,'diagnostics':diagnostics,'sanity':sanity}

def evaluate_lcc(result,lcc_loader,resources,device,use_amp):
    name=result['name'];progress(name,'LCC external-development evaluation begins')
    model=result['model'].to(device)
    logits,y,keys,pred=eval_lcc_spatial(model,lcc_loader,device,use_amp,
        resources['lcc_df'],result['locked_threshold'])
    score=binary_score(torch.as_tensor(logits)).cpu().numpy()
    candidate_n=len(resources['lcc_df'])
    metrics=metrics_binary(y,score,result['locked_threshold'],
        coverage=len(y)/candidate_n,candidate_n=candidate_n)
    pred['locked_threshold']=result['locked_threshold']
    pred.to_csv(result['out_dir']/'lcc_predictions.csv',index=False)
    result['lcc']=metrics
    model.cpu();torch.cuda.empty_cache()
    progress(name,f'LCC HTER={metrics["hter"]:.6f}, AUC={metrics["auc"]:.6f}, coverage={metrics["detector_coverage"]:.5f}')
    return result

def freeze_gate(vanilla,csmr):
    delta={'val_acer':csmr['val']['acer']-vanilla['val']['acer'],
        'val_auc':csmr['val']['auc']-vanilla['val']['auc'],
        'lcc_apcer':csmr['lcc']['apcer']-vanilla['lcc']['apcer'],
        'lcc_bpcer':csmr['lcc']['bpcer']-vanilla['lcc']['bpcer'],
        'lcc_hter':csmr['lcc']['hter']-vanilla['lcc']['hter'],
        'lcc_auc':csmr['lcc']['auc']-vanilla['lcc']['auc']}
    primary=delta['lcc_auc']>=.015 and delta['val_acer']<=.010
    secondary=(delta['lcc_hter']<=-.015 and delta['lcc_auc']>=0 and delta['val_acer']<=.010)
    gate='PRIMARY_AUC_GATE' if primary else 'SECONDARY_HTER_GATE' if secondary else 'NONE'
    return ('PASS' if primary or secondary else 'FAIL'),gate,delta

def paired_table(vanilla,csmr,split):
    if split=='development':
        rows=[('Val ACER','val','acer'),('Val AUC','val','auc'),
              ('LCC APCER','lcc','apcer'),('LCC BPCER','lcc','bpcer'),
              ('LCC HTER','lcc','hter'),('LCC AUC','lcc','auc')]
    else:
        rows=[('Test APCER','test','apcer'),('Test BPCER','test','bpcer'),
              ('Test ACER','test','acer'),('Test AUC','test','auc')]
    text=['| Metric | Vanilla seed100 | CSMR seed100 | Delta (CSMR - Vanilla) |',
          '|---|---:|---:|---:|']
    for label,field,metric in rows:
        a=vanilla[field][metric];b=csmr[field][metric]
        text.append(f'| {label} | {a:.6f} | {b:.6f} | {b-a:+.6f} |')
    return '\n'.join(text)


In [22]:
# Verbatim E1 v5.3 generator functions from e1_v5_3_training.ipynb.
# Defined before development; executed ONLY after a persisted PASS freeze gate.
SCRFD_BBOX_EXPANSION_FACTOR = 1.55
CELEBA_FALLBACK_EXPANSION_FACTOR = 1.50
TRAIN_MIN_FACE_SIZE = 48
SCRFD_PRIMARY_INPUT_SIZE = (640, 640)
SCRFD_FALLBACK_INPUT_SIZES = [(480, 480), (320, 320)]
SCRFD_CONF_THRESH = 0.50
RELIABLE_RAW_IOU_MIN = 0.50
RELIABLE_CENTER_SHIFT_MAX = 0.20
DATA_ROOT = CELEBA_ROOT
E1_GENERATOR_NOTEBOOK_SHA256 = "971d4bbf0ff709ab073b0a7266f26132cf7e23cc05ad73bf08f86d3ff509d539"

def bbox_wh(b):
    x1, y1, x2, y2 = map(float, b)
    return np.array([x2 - x1, y2 - y1], dtype=np.float64)


def bbox_area(b):
    w, h = bbox_wh(b)
    return max(0.0, float(w)) * max(0.0, float(h))


def bbox_center(b):
    x1, y1, x2, y2 = map(float, b)
    return np.array([(x1 + x2) / 2.0, (y1 + y2) / 2.0], dtype=np.float64)


def bbox_intersection_area(a, b):
    ax1, ay1, ax2, ay2 = map(float, a)
    bx1, by1, bx2, by2 = map(float, b)
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    return max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)


def bbox_iou(a, b):
    inter = bbox_intersection_area(a, b)
    union = bbox_area(a) + bbox_area(b) - inter
    return 0.0 if union <= 0 else float(inter / union)


def normalized_center_shift(reference, candidate):
    ref_wh = bbox_wh(reference)
    denom = max(float(ref_wh[0]), float(ref_wh[1]), 1e-8)
    return float(np.linalg.norm(bbox_center(reference) - bbox_center(candidate)) / denom)


def read_celeba_bbox_xyxy(img_path, image_shape=None):
    """Convert CelebA-Spoof 224-reference *_BB.txt coordinates to native-image xyxy."""
    if image_shape is None:
        image = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(f"Cannot read image: {img_path}")
        h_img, w_img = image.shape[:2]
    else:
        h_img, w_img = image_shape[:2]

    bb_path = os.path.splitext(img_path)[0] + "_BB.txt"
    if not os.path.exists(bb_path):
        raise FileNotFoundError(f"Missing bbox file: {bb_path}")

    with open(bb_path, "r") as f:
        parts = f.readline().strip().split()
    if len(parts) < 4:
        raise ValueError(f"bbox needs >=4 values, got {parts}")

    x, y, w, h = map(float, parts[:4])
    x = x / 224.0 * w_img
    y = y / 224.0 * h_img
    w = w / 224.0 * w_img
    h = h / 224.0 * h_img
    if w <= 0 or h <= 0:
        raise ValueError(f"Non-positive CelebA bbox at {bb_path}: {(x, y, w, h)}")
    return np.array([x, y, x + w, y + h], dtype=np.float64)


def _select_target_detection(bboxes, gt_bbox):
    if bboxes is None or len(bboxes) == 0:
        return None

    candidates = []
    for row in bboxes:
        det = np.asarray(row[:4], dtype=np.float64)
        conf = float(row[4])
        candidates.append({
            "bbox": det,
            "confidence": conf,
            "raw_iou": bbox_iou(gt_bbox, det),
            "center_shift": normalized_center_shift(gt_bbox, det),
            "n_faces": int(len(bboxes)),
        })

    return max(candidates, key=lambda x: (x["raw_iou"], x["confidence"]))


def _scrfd_detect(detector, image_bgr, input_size):
    """
    Prefer detect(..., input_size=...). Older InsightFace variants are handled
    by temporarily changing detector.input_size.
    """
    try:
        return detector.detect(
            image_bgr,
            input_size=tuple(input_size),
            max_num=0,
            metric="default",
        )
    except TypeError:
        old_input_size = getattr(detector, "input_size", None)
        detector.input_size = tuple(input_size)
        try:
            return detector.detect(image_bgr, max_num=0, metric="default")
        finally:
            detector.input_size = old_input_size


def build_cache_record(detector, rel_path):
    img_path = os.path.join(DATA_ROOT, rel_path)
    image = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if image is None:
        return {"status": "INVALID", "reason": "image_read_failure"}

    try:
        gt = read_celeba_bbox_xyxy(img_path, image.shape)
    except Exception as exc:
        return {
            "status": "INVALID",
            "reason": f"celeba_bbox_error:{type(exc).__name__}",
        }

    gt_w, gt_h = bbox_wh(gt)
    gt_min_side = float(min(gt_w, gt_h))

    sizes_to_try = [SCRFD_PRIMARY_INPUT_SIZE] + list(SCRFD_FALLBACK_INPUT_SIZES)
    best_unreliable = None

    for input_size in sizes_to_try:
        bboxes, _ = _scrfd_detect(detector, image, input_size=input_size)
        candidate = _select_target_detection(bboxes, gt)
        if candidate is None:
            continue

        candidate["input_size"] = list(input_size)

        if (
            candidate["raw_iou"] >= RELIABLE_RAW_IOU_MIN
            and candidate["center_shift"] <= RELIABLE_CENTER_SHIFT_MAX
        ):
            det = candidate["bbox"]
            dw, dh = bbox_wh(det)
            return {
                "status": "VALID",
                "bbox_source": f"SCRFD_{input_size[0]}",
                "bbox_xyxy": [float(x) for x in det],
                "crop_factor": SCRFD_BBOX_EXPANSION_FACTOR,
                "face_min_side": float(min(dw, dh)),
                "confidence": candidate["confidence"],
                "n_faces": candidate["n_faces"],
                "raw_iou_to_celeba": candidate["raw_iou"],
                "center_shift_norm": candidate["center_shift"],
                "det_input_size": list(input_size),
                "gt_min_side": gt_min_side,
            }

        if best_unreliable is None or (
            candidate["raw_iou"], candidate["confidence"]
        ) > (
            best_unreliable["raw_iou"], best_unreliable["confidence"]
        ):
            best_unreliable = candidate

    # Do not auto-drop detector failures: the diagnostic showed class-dependent
    # failure, especially for Physical Spoof. Keep sample through CelebA fallback.
    if best_unreliable is None:
        return {
            "status": "FALLBACK_DET_FAIL",
            "bbox_source": "CELEBA_FALLBACK_DET_FAIL",
            "bbox_xyxy": [float(x) for x in gt],
            "crop_factor": CELEBA_FALLBACK_EXPANSION_FACTOR,
            "face_min_side": gt_min_side,
            "confidence": None,
            "n_faces": 0,
            "raw_iou_to_celeba": None,
            "center_shift_norm": None,
            "det_input_size": None,
            "gt_min_side": gt_min_side,
        }

    # Suspicious geometry: retain Val/Test by GT fallback, but Train filters it.
    return {
        "status": "FALLBACK_SUSPICIOUS",
        "bbox_source": "CELEBA_FALLBACK_SUSPICIOUS",
        "bbox_xyxy": [float(x) for x in gt],
        "crop_factor": CELEBA_FALLBACK_EXPANSION_FACTOR,
        "face_min_side": gt_min_side,
        "confidence": best_unreliable["confidence"],
        "n_faces": best_unreliable["n_faces"],
        "raw_iou_to_celeba": best_unreliable["raw_iou"],
        "center_shift_norm": best_unreliable["center_shift"],
        "det_input_size": best_unreliable["input_size"],
        "gt_min_side": gt_min_side,
    }


In [23]:
# Development phase. No official Test metadata or Test image is opened here.
check_fixed_contract()
progress('PREFLIGHT','CSMR-v1 FREEZE CHECK | paired replication | training seed=100 | frozen Stage-0 seed42 manifest')
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU accelerator is required for the paired freeze check')
device=torch.device('cuda')
use_amp=bool(AMP)
progress('PREFLIGHT',f'GPU/AMP: {gpu_usage()} | AMP={use_amp}')
progress('PREFLIGHT','Loading frozen Stage 0 resources by direct configured path')
resources=load_stage0_resources()
progress('PREFLIGHT',f'Train={len(resources["train_keys"])} Val={len(resources["val_keys"])} LCC valid={len(resources["lcc_valid_paths"])}')
progress('PREFLIGHT',f'Stage 0 hashes: {resources["fingerprints"]}')
progress('PREFLIGHT','Checking Stage 1 historical reference summary/hash only; its PAD checkpoint is never loaded')
stage1_reference,stage1_path=load_stage1_reference(resources)
progress('PREFLIGHT',f'Stage 1 summary={stage1_path} checkpoint SHA={stage1_reference["checkpoint_sha256"]}')
celeba_root=find_celeba_root()
lcc_root=find_lcc_root(resources['stage0_dir']/'lcc_external_dev_manifest.csv')
progress('PREFLIGHT',f'Raw roots: CelebA={celeba_root}; LCC={lcc_root}')
assert_raw_image_paths(resources,celeba_root,lcc_root)
progress('PREFLIGHT','All frozen development raw image paths exist')
loader_args={'batch_size':BATCH_SIZE,'shuffle':False,'drop_last':False,
             'num_workers':NUM_WORKERS,'pin_memory':True,'persistent_workers':NUM_WORKERS>0}
val_ds=CelebAMicroDataset(resources['val_keys'],resources['val_labels'],
    resources['val_codes'],celeba_root,resources['bbox_cache'],resources['stage0_dir'],train=False)
lcc_ds=LCCValidDataset(resources['lcc_df'],lcc_root,resources['lcc_cache'],resources['stage0_dir'])
val_loader=DataLoader(val_ds,**loader_args)
lcc_loader=DataLoader(lcc_ds,**loader_args)
masks,_=radial_masks(device=device)
progress('PREFLIGHT',f'ImageNet weights source: {IMAGENET_WEIGHTS_PATH or "torchvision pretrained policy"}')
imagenet_state=load_imagenet_weights(IMAGENET_WEIGHTS_PATH)
progress('PREFLIGHT','ImageNet weights loaded. Starting Vanilla then CSMR; no LCC during epochs.')
output_root=Path(OUTPUT_ROOT)
if output_root.exists():
    raise RuntimeError(f'Unsafe output directory collision: {output_root}; start a clean session')
output_root.mkdir(parents=True)
comparison_dir=output_root/'comparison';comparison_dir.mkdir()
strict_json_write(comparison_dir/'resource_fingerprints.json',resources['fingerprints'])
vanilla=train_one('vanilla_seed100',False,resources,celeba_root,imagenet_state,
    val_loader,device,use_amp,masks,output_root)
csmr=train_one('csmr_seed100',True,resources,celeba_root,imagenet_state,
    val_loader,device,use_amp,masks,output_root)
progress('DEVELOPMENT','Both best checkpoints selected and Val thresholds locked; starting LCC')
vanilla=evaluate_lcc(vanilla,lcc_loader,resources,device,use_amp)
csmr=evaluate_lcc(csmr,lcc_loader,resources,device,use_amp)
progress('DEVELOPMENT','Both LCC evaluations completed; Test remains unopened')


[2026-09-29T12:45:55+00:00 | +0.3s | PREFLIGHT] CSMR-v1 FREEZE CHECK | paired replication | training seed=100 | frozen Stage-0 seed42 manifest
[2026-09-29T12:45:56+00:00 | +0.6s | PREFLIGHT] GPU/AMP: GPU Tesla T4 | allocated 0 MiB | reserved 0 MiB | peak 0 MiB | AMP=True
[2026-09-29T12:45:56+00:00 | +0.6s | PREFLIGHT] Loading frozen Stage 0 resources by direct configured path
[2026-09-29T12:45:56+00:00 | +1.2s | PREFLIGHT] Train=5000 Val=2000 LCC valid=3762
[2026-09-29T12:45:56+00:00 | +1.2s | PREFLIGHT] Stage 0 hashes: {'parent_e1_manifest_sha256': '7799486ebcae0babe09e061579ac05db7c1dc61e10746be0755bcddc10b2df9e', 'parent_e1_train_fingerprint': '4ecbcee78758366dd454c0e446444f6c74c82ac5c4d1c937730382b34d4e25f8', 'parent_e1_val_fingerprint': '4560d12efe0bb840c0584a38bda12097e2048f3066d89177cda62f8d931edf91', 'micro_manifest_sha256': '7c1eed4abaab7ee193758ea4bf4a0c36b98e43b5fb8d84c5c0e7d424607a0aa3', 'micro_train_keys_fingerprint': '9ce47155516d265dc9d6eccbc0ea4105684afe7c9f87ae00cb5378

100%|██████████| 9.83M/9.83M [00:00<00:00, 121MB/s]

[2026-09-29T12:47:32+00:00 | +97.3s | PREFLIGHT] ImageNet weights loaded. Starting Vanilla then CSMR; no LCC during epochs.
[2026-09-29T12:47:32+00:00 | +97.3s | vanilla_seed100] Seeding training RNG=100 before model and DataLoader construction


[2026-09-29T12:47:32+00:00 | +97.4s | vanilla_seed100] Independent ImageNet model; params=1108515; GPU Tesla T4 | allocated 5 MiB | reserved 6 MiB | peak 5 MiB
[2026-09-29T12:47:32+00:00 | +97.4s | vanilla_seed100] Train batches=40; logging every 2 batches
[2026-09-29T12:47:32+00:00 | +97.4s | vanilla_seed100] Epoch 1/6 start; GPU Tesla T4 | allocated 5 MiB | reserved 6 MiB | peak 5 MiB


/tmp/ipykernel_58/2541015953.py:69: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_58/2541015953.py:117: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


...

/tmp/ipykernel_58/2815326475.py:584: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


[2026-09-29T12:55:22+00:00 | +567.5s | vanilla_seed100] LCC HTER=0.340274, AUC=0.690181, coverage=0.99894
[2026-09-29T12:55:22+00:00 | +567.5s | csmr_seed100] LCC external-development evaluation begins


/tmp/ipykernel_58/2815326475.py:614: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ True False  True ... False False False]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  full.loc[valid, "pred_real"] = full.loc[valid, "pad_logit_score"] >= float(threshold)
/tmp/ipykernel_58/2815326475.py:584: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


[2026-09-29T12:55:51+00:00 | +596.4s | csmr_seed100] LCC HTER=0.331171, AUC=0.714238, coverage=0.99894
[2026-09-29T12:55:51+00:00 | +596.4s | DEVELOPMENT] Both LCC evaluations completed; Test remains unopened


/tmp/ipykernel_58/2815326475.py:614: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ True False  True ...  True False False]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  full.loc[valid, "pred_real"] = full.loc[valid, "pad_logit_score"] >= float(threshold)


In [24]:
# The rule lives in freeze_gate() above and is fixed before training.
FREEZE_CHECK,triggered_gate,paired_delta=freeze_gate(vanilla,csmr)
manifest_sha=resources['fingerprints']['micro_manifest_sha256']
pretest={'schema':'csmr_v1_seed100_freeze_pretest_v1',
    'training_seed':100,
    'frozen_development_manifest':{'identity':'micro_manifest_v1_seed42.npz','sha256':manifest_sha},
    'resource_fingerprints':resources['fingerprints'],
    'gate_definitions':{
        'primary_rule':'delta_lcc_auc >= +0.015 AND delta_val_acer <= +0.010',
        'secondary_rule':'delta_lcc_hter <= -0.015 AND delta_lcc_auc >= 0 AND delta_val_acer <= +0.010',
        'delta_definition':'CSMR minus Vanilla'},
    'freeze_gate':{'result':FREEZE_CHECK,'triggered_gate':triggered_gate},
    'vanilla':{'val':vanilla['val'],'lcc':vanilla['lcc'],
        'selected_epoch':vanilla['selected_epoch'],
        'locked_val_threshold':vanilla['locked_threshold'],
        'checkpoint_sha256':vanilla['checkpoint_sha256']},
    'csmr':{'val':csmr['val'],'lcc':csmr['lcc'],
        'selected_epoch':csmr['selected_epoch'],
        'locked_val_threshold':csmr['locked_threshold'],
        'checkpoint_sha256':csmr['checkpoint_sha256']},
    'paired_delta':paired_delta,
    'stage1_historical_reference_summary_sha256':sha256_file(stage1_path),
    'celeba_test2000_opened':False}
pretest_path=comparison_dir/'freeze_check_seed100_summary_pretest.json'
strict_json_write(pretest_path,pretest)
table=paired_table(vanilla,csmr,'development')
lines=['# Seed-100 paired freeze decision — PRETEST','',
    'Recorded before reading official CelebA Test metadata or images. LCC is external development. The gate is a fixed heuristic, not a significance test.','',
    '## Frozen protocol',
    f'Training seed 100; frozen Stage 0 seed42 manifest SHA `{manifest_sha}`; Train/Val 5,000/2,000. Independent ImageNet initialization, same architecture, crop, normalization, optimizer, budget and Val-only checkpoint/threshold policy. Stage 1 checkpoint was not loaded.','',
    '## Paired development metrics',table,'',
    f'## FREEZE CHECK = {FREEZE_CHECK}',
    f'Triggered gate: `{triggered_gate}`. Primary: {pretest["gate_definitions"]["primary_rule"]}. Secondary: {pretest["gate_definitions"]["secondary_rule"]}.','',
    '## Provenance',
    f'Vanilla: epoch {vanilla["selected_epoch"]}, threshold {vanilla["locked_threshold"]:.8f}, SHA `{vanilla["checkpoint_sha256"]}`.',
    f'CSMR: epoch {csmr["selected_epoch"]}, threshold {csmr["locked_threshold"]:.8f}, SHA `{csmr["checkpoint_sha256"]}`.',
    f'Resource fingerprints: `{json.dumps(resources["fingerprints"],sort_keys=True)}`.','']
pretest_report_path=comparison_dir/'freeze_check_seed100_report_pretest.md'
pretest_report_path.write_text('\n'.join(lines))
metrics_rows=[]
for label,split,metric in [('Val ACER','val','acer'),('Val AUC','val','auc'),
    ('LCC APCER','lcc','apcer'),('LCC BPCER','lcc','bpcer'),
    ('LCC HTER','lcc','hter'),('LCC AUC','lcc','auc')]:
    metrics_rows.append({'metric':label,'vanilla_seed100':vanilla[split][metric],
        'csmr_seed100':csmr[split][metric],
        'delta_csmr_minus_vanilla':csmr[split][metric]-vanilla[split][metric]})
pd.DataFrame(metrics_rows).to_csv(comparison_dir/'paired_metrics.csv',index=False)
progress('FREEZE',f'PRETEST decision persisted: {pretest_path} and {pretest_report_path}')
print(table)
progress('FREEZE',f'FREEZE CHECK = {FREEZE_CHECK}; triggered gate = {triggered_gate}')
if FREEZE_CHECK=='FAIL':
    progress('FREEZE','Freeze check failed; held-out CelebA Test remains untouched.')


[2026-09-29T12:55:51+00:00 | +596.5s | FREEZE] PRETEST decision persisted: /kaggle/working/csmr_v1_freeze_check_seed100/comparison/freeze_check_seed100_summary_pretest.json and /kaggle/working/csmr_v1_freeze_check_seed100/comparison/freeze_check_seed100_report_pretest.md
| Metric | Vanilla seed100 | CSMR seed100 | Delta (CSMR - Vanilla) |
|---|---:|---:|---:|
| Val ACER | 0.021062 | 0.021357 | +0.000295 |
| Val AUC | 0.997817 | 0.997508 | -0.000309 |
| LCC APCER | 0.387489 | 0.412985 | +0.025497 |
| LCC BPCER | 0.293059 | 0.249357 | -0.043702 |
| LCC HTER | 0.340274 | 0.331171 | -0.009103 |
| LCC AUC | 0.690181 | 0.714238 | +0.024057 |
[2026-09-29T12:55:51+00:00 | +596.5s | FREEZE] FREEZE CHECK = PASS; triggered gate = PRIMARY_AUC_GATE


In [30]:
from pathlib import Path
import hashlib
import shutil

source = Path(
        "/kaggle/input/datasets/maithanhphuong/"
        "scrfd-model"
    )
source = dataset_dir / "det_500m.onnx"

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

assert source.is_file(), f"Không tìm thấy model: {source}"
expected = resources["fingerprints"]["lcc_detector_sha256"]
assert sha256(source) == expected, "Model SCRFD không khớp Stage 0"

target.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(source, target)
assert sha256(target) == expected
print("SCRFD sẵn sàng:", target)

SCRFD sẵn sàng: /root/.insightface/models/buffalo_s/det_500m.onnx


In [32]:
%pip install -q "insightface==2.0" "onnxruntime"

import insightface
import onnxruntime as ort

print("InsightFace:", insightface.__version__)
print("ONNX Runtime providers:", ort.get_available_providers())

assert insightface.__version__ == source_policy["insightface_version"]
assert "CPUExecutionProvider" in ort.get_available_providers()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 13.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 54.4 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.
InsightFace: 2.0
ONNX Runtime providers: ['AzureExecutionProvider', 'CPUExecutionProvider']


In [33]:
# This cell is the ONLY place that may read official CelebA Test metadata/images.
celeba_test_opened=False
celeba_test2000=None
if FREEZE_CHECK!='PASS':
    progress('TEST','Freeze check failed; held-out CelebA Test remains untouched.')
else:
    if not pretest_path.is_file() or not pretest_report_path.is_file():
        raise RuntimeError('Pretest freeze decision files are missing; refusing to open Test')
    persisted=strict_json_load(pretest_path)
    if persisted['freeze_gate']['result']!='PASS' or persisted['resource_fingerprints']!=resources['fingerprints']:
        raise RuntimeError('Persisted freeze gate/resource mismatch; refusing to open Test')
    progress('TEST','Pretest PASS decision verified on disk')
    test_metadata_path=Path(CELEBA_ROOT)/'metas/intra_test/test_label.json'
    if not test_metadata_path.is_file():
        raise FileNotFoundError(f'Official CelebA Test metadata missing: {test_metadata_path}; Test remains unopened')
    # Mark Test OPENED before reading any official Test metadata.
    marker_path=comparison_dir/'celeba_test2000_opened_marker.json'
    strict_json_write(marker_path,{'schema':'csmr_v1_test_opened_marker_v1',
        'opened_at_utc':datetime.now(timezone.utc).isoformat(),
        'pretest_summary_sha256':sha256_file(pretest_path),
        'metadata_path':str(test_metadata_path),
        'test_subset_seed':TEST_SUBSET_SEED,'test_subset_n':TEST_SUBSET_N})
    celeba_test_opened=True
    progress('TEST',f'POST-FREEZE held-out in-domain confirmation OPENED; marker={marker_path}')
    official_meta=strict_json_load(test_metadata_path)
    if not isinstance(official_meta,dict) or len(official_meta)<TEST_SUBSET_N:
        raise RuntimeError('Official Test metadata malformed or too small for Test-2000')
    official_keys=sorted(map(str,official_meta))
    if len(official_keys)!=len(set(official_keys)) or not all(k.startswith('Data/test/') for k in official_keys):
        raise RuntimeError('Official Test metadata keys do not unambiguously identify Data/test split')
    def official_code(key):
        row=official_meta[key]
        if not isinstance(row,(list,tuple)) or len(row)<=40:
            raise RuntimeError(f'Official Test metadata lacks E1 attack code at index 40: {key}')
        value=row[40]
        if isinstance(value,bool) or int(value)!=value or int(value)<0:
            raise RuntimeError(f'Invalid official Test attack code: {key}')
        return int(value)
    official_codes=np.asarray([official_code(k) for k in official_keys],dtype=np.int64)
    official_labels=labels_from_attack_codes(official_codes)
    if set(map(int,official_labels))!={0,1,2}:
        raise RuntimeError('Official Test metadata cannot provide all three E1 PAD classes')
    if len(official_keys)==TEST_SUBSET_N:
        selected_keys=list(official_keys)
    else:
        selected_keys,_=train_test_split(official_keys,train_size=TEST_SUBSET_N,
            stratify=official_labels,random_state=TEST_SUBSET_SEED)
    if len(selected_keys)!=TEST_SUBSET_N or len(set(selected_keys))!=TEST_SUBSET_N:
        raise RuntimeError('Deterministic official Test-2000 subset has invalid size/duplicates')
    dev_keys=set(map(str,resources['train_keys']))|set(map(str,resources['val_keys']))
    if dev_keys.intersection(selected_keys):
        raise RuntimeError('Official Test-2000 overlaps frozen micro-Train/Val')
    selected_codes=np.asarray([official_code(k) for k in selected_keys],dtype=np.int64)
    selected_labels=labels_from_attack_codes(selected_codes)
    selected_binary=(selected_labels==0).astype(np.int64)
    manifest=pd.DataFrame({'sample_key':selected_keys,
        'binary_label_real':selected_binary,
        'three_class_label':selected_labels,
        'attack_code':selected_codes,
        'official_split':'test','sampling_seed':TEST_SUBSET_SEED})
    if len(manifest)!=TEST_SUBSET_N:
        raise RuntimeError('Test manifest is not exactly 2000 rows')
    test_manifest_path=comparison_dir/'celeba_test2000_manifest.csv'
    manifest.to_csv(test_manifest_path,index=False)
    manifest_sha=ordered_path_fingerprint(selected_keys)
    test_metadata_sha=sha256_file(test_metadata_path)
    progress('TEST',f'Selected {len(selected_keys)} official Test keys; stratified on 3 E1 PAD classes; class counts={dict(Counter(map(int,selected_labels)))}; ordered SHA={manifest_sha}')
    # Generate E1 v5.3 bbox records for exactly the fixed selected Test keys.
    progress('TEST','Building Test-only E1 v5.3 SCRFD bbox cache for exactly 2,000 selected keys')
    stage0_bbox_payload=strict_json_load(resources['stage0_dir']/'celeba_micro_bbox_cache.json')
    source_policy=stage0_bbox_payload.get('policy',{})
    fixed_policy={'detector':'SCRFD-500M / buffalo_s detection',
        'detector_model':'det_500m.onnx',
        'primary_input_size':[640,640],
        'fallback_input_sizes':[[480,480],[320,320]],
        'det_thresh':.50,'reliable_raw_iou_min':.50,
        'reliable_center_shift_max':.20,'scrfd_crop_factor':1.55,
        'celeba_fallback_crop_factor':1.50,'train_min_face_px':48}
    for field,expected in fixed_policy.items():
        if source_policy.get(field)!=expected:
            raise RuntimeError(f'Stage 0/E1 detector policy mismatch: {field}: {source_policy.get(field)}')
    detector_path=Path(SCRFD_MODEL_ROOT).expanduser()/'models/buffalo_s/det_500m.onnx'
    if not detector_path.is_file():
        raise FileNotFoundError(f'Exact SCRFD model missing: {detector_path}; attach frozen buffalo_s model')
    detector_sha=sha256_file(detector_path)
    if detector_sha!=resources['fingerprints']['lcc_detector_sha256']:
        raise RuntimeError('SCRFD detector SHA differs from frozen Stage 0 detector SHA')
    try:
        import insightface
        import onnxruntime as ort
        from insightface.app import FaceAnalysis
    except ImportError as exc:
        raise RuntimeError('Exact E1 SCRFD cache build needs InsightFace and ONNX Runtime; install compatible versions before rerunning from a clean session') from exc
    insightface_version=getattr(insightface,'__version__','unknown')
    if insightface_version!=source_policy.get('insightface_version'):
        raise RuntimeError(f'InsightFace version differs from Stage 0 policy: {insightface_version} vs {source_policy.get("insightface_version")}')
    providers=ort.get_available_providers()
    if 'CUDAExecutionProvider' in providers:
        scrfd_providers=['CUDAExecutionProvider','CPUExecutionProvider'];scrfd_ctx_id=0
    elif 'CPUExecutionProvider' in providers:
        scrfd_providers=['CPUExecutionProvider'];scrfd_ctx_id=-1
    else:
        raise RuntimeError(f'No compatible ONNX Runtime provider: {providers}')
    progress('TEST',f'SCRFD SHA={detector_sha}; InsightFace={insightface_version}; providers={scrfd_providers}')
    face_app=FaceAnalysis(name='buffalo_s',root=str(Path(SCRFD_MODEL_ROOT).expanduser()),
        allowed_modules=['detection'],providers=scrfd_providers)
    face_app.prepare(ctx_id=scrfd_ctx_id,det_size=SCRFD_PRIMARY_INPUT_SIZE,det_thresh=SCRFD_CONF_THRESH)
    detector=face_app.models.get('detection')
    if detector is None:
        raise RuntimeError('SCRFD detection model absent from buffalo_s')
    test_records={}
    for index,key in enumerate(selected_keys,1):
        test_records[key]=build_cache_record(detector,key)
        if index==1 or index%100==0 or index==TEST_SUBSET_N:
            progress('TEST',f'SCRFD bbox {index}/{TEST_SUBSET_N}; status={test_records[key].get("status")}')
    del detector,face_app
    torch.cuda.empty_cache()
    test_bbox_cache=test_records
    test2000_manifest=manifest.set_index('sample_key')
    if set(test_bbox_cache.keys())!=set(test2000_manifest.index.astype(str)) or len(test_bbox_cache)!=TEST_SUBSET_N:
        raise RuntimeError('Test bbox cache keys differ from exact Test-2000 manifest')
    status_counts=Counter(str(rec.get('status','')) for rec in test_records.values())
    invalid_n=sum(v for k,v in status_counts.items() if k not in CELEBA_BBOX_STATUS_FACTORS)
    cache_payload={'schema':'celeba_test2000_bbox_cache_v1',
        'policy':source_policy,'records':test_records,
        'selected_n':TEST_SUBSET_N,'ordered_path_fingerprint_sha256':manifest_sha,
        'official_test_metadata_sha256':test_metadata_sha,
        'detector_sha256':detector_sha,'insightface_version':insightface_version,
        'onnxruntime_version':ort.__version__,'onnx_providers':scrfd_providers,
        'e1_generator_notebook_sha256':E1_GENERATOR_NOTEBOOK_SHA256,
        'source_stage0_bbox_cache_sha256':resources['fingerprints']['celeba_bbox_cache_sha256'],
        'status_counts':dict(status_counts),'valid_n':int(status_counts.get('VALID',0)),
        'fallback_det_fail_n':int(status_counts.get('FALLBACK_DET_FAIL',0)),
        'fallback_suspicious_n':int(status_counts.get('FALLBACK_SUSPICIOUS',0)),
        'invalid_failure_n':int(invalid_n)}
    test_cache_path=comparison_dir/'celeba_test2000_bbox_cache.json'
    strict_json_write(test_cache_path,cache_payload)
    test_cache_sha=sha256_file(test_cache_path)
    progress('TEST',f'Saved Test-only bbox cache {test_cache_path}; SHA={test_cache_sha}; statuses={dict(status_counts)}')
    if invalid_n:
        failed=[(k,v.get('reason',v.get('status'))) for k,v in test_records.items() if v.get('status') not in CELEBA_BBOX_STATUS_FACTORS]
        raise RuntimeError(f'Test bbox generator produced {invalid_n} INVALID/failure records; first={failed[:3]}; cache saved for audit, no resampling')
    for key in selected_keys:
        validate_bbox_record(test_records[key],'celeba',key)
    missing_images=[k for k in selected_keys if not (Path(CELEBA_ROOT)/k).is_file()]
    if missing_images:
        raise FileNotFoundError(f'{len(missing_images)} selected official Test images missing; first={missing_images[:3]}. Do not resample.')
    progress('TEST',f'All {TEST_SUBSET_N} Test images and frozen E1 bbox records valid; cache SHA={sha256_file(test_cache_path)}')
    test_ds=CelebAMicroDataset(selected_keys,selected_labels,selected_codes,
        CELEBA_ROOT,test_records,resources['stage0_dir'],train=False)
    test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,drop_last=False,
        num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=NUM_WORKERS>0)
    test_rows=[]
    for result in (vanilla,csmr):
        name=result['name'];path=result['out_dir']/'best.pth'
        if sha256_file(path)!=result['checkpoint_sha256']:
            raise RuntimeError(f'Frozen {name} checkpoint SHA changed before Test')
        state=torch.load(path,map_location='cpu',weights_only=False)
        model=result['model']
        model.load_state_dict(state['state_dict'],strict=True)
        model.to(device)
        progress('TEST',f'Evaluating {name} on the SAME fixed Test-2000 manifest with its locked Val threshold')
        logits,y,keys,codes=eval_spatial(model,test_loader,device,use_amp)
        assert_ordered_keys(keys,selected_keys,name+' Test-2000')
        if not np.array_equal(y,selected_labels) or not np.array_equal(codes,selected_codes):
            raise RuntimeError(f'{name} Test labels/codes differ from fixed manifest')
        metrics,_=metrics_celeba(y,logits,result['locked_threshold'])
        if metrics['n_scored']!=TEST_SUBSET_N or metrics['detector_coverage']!=1.:
            raise RuntimeError(f'{name} Test coverage is incomplete; do not replace samples')
        make_prediction_frame(keys,y,codes,logits,result['locked_threshold']).to_csv(
            comparison_dir/f'{name}_test_predictions.csv',index=False)
        result['test']=metrics
        model.cpu();torch.cuda.empty_cache()
        progress('TEST',f'{name}: APCER={metrics["apcer"]:.6f} BPCER={metrics["bpcer"]:.6f} ACER={metrics["acer"]:.6f} AUC={metrics["auc"]:.6f}')
    test_table=paired_table(vanilla,csmr,'test')
    print('POST-FREEZE HELD-OUT IN-DOMAIN CONFIRMATION\n'+test_table)
    for metric in ('apcer','bpcer','acer','auc'):
        test_rows.append({'metric':'Test '+metric.upper(),
            'vanilla_seed100':vanilla['test'][metric],
            'csmr_seed100':csmr['test'][metric],
            'delta_csmr_minus_vanilla':csmr['test'][metric]-vanilla['test'][metric]})
    pd.DataFrame(test_rows).to_csv(comparison_dir/'celeba_test2000_metrics.csv',index=False)
    celeba_test2000={'schema':'csmr_v1_celeba_test2000_v1',
        'role':'post-freeze held-out in-domain descriptive confirmation',
        'official_metadata_sha256':test_metadata_sha,
        'test_bbox_cache_sha256':test_cache_sha,
        'test_bbox_cache_status_counts':dict(status_counts),
        'test_bbox_generator_provenance':{'e1_notebook_sha256':E1_GENERATOR_NOTEBOOK_SHA256,
            'detector_sha256':detector_sha,'insightface_version':insightface_version,
            'onnxruntime_version':ort.__version__,'providers':scrfd_providers},
        'ordered_path_fingerprint_sha256':manifest_sha,
        'test_subset_seed':TEST_SUBSET_SEED,'selected_n':TEST_SUBSET_N,
        'stratification':'three-class E1 attack-code mapping from official metadata index 40',
        'class_counts':{str(k):int(v) for k,v in Counter(map(int,selected_labels)).items()},
        'valid_coverage':1.0,'manifest_file':test_manifest_path.name,
        'vanilla':vanilla['test'],'csmr':csmr['test'],
        'delta_csmr_minus_vanilla':{m:csmr['test'][m]-vanilla['test'][m]
            for m in ('apcer','bpcer','acer','auc')},
        'freeze_gate_result_unchanged':FREEZE_CHECK}
    strict_json_write(comparison_dir/'celeba_test2000_summary.json',celeba_test2000)
    progress('TEST','Saved Test-2000 manifest, both predictions, metrics and summary; freeze decision unchanged')


[2026-09-29T13:07:49+00:00 | +1,314.2s | TEST] Pretest PASS decision verified on disk
[2026-09-29T13:07:49+00:00 | +1,314.2s | TEST] POST-FREEZE held-out in-domain confirmation OPENED; marker=/kaggle/working/csmr_v1_freeze_check_seed100/comparison/celeba_test2000_opened_marker.json
[2026-09-29T13:07:50+00:00 | +1,314.8s | TEST] Selected 2000 official Test keys; stratified on 3 E1 PAD classes; class counts={1: 1057, 0: 593, 2: 350}; ordered SHA=df3416273c3e7d8bf0caa3b010454ee73236211e034ec12bfb481f056fe237b8
[2026-09-29T13:07:50+00:00 | +1,314.8s | TEST] Building Test-only E1 v5.3 SCRFD bbox cache for exactly 2,000 selected keys
[2026-09-29T13:07:50+00:00 | +1,314.9s | TEST] SCRFD SHA=5e4447f50245bbd7966bd6c0fa52938c61474a04ec7def48753668a9d8b4ea3a; InsightFace=2.0; providers=['CPUExecutionProvider']
[2026-09-29T13:07:50+00:00 | +1,315.1s | TEST] SCRFD bbox 1/2000; status=VALID
[2026-09-29T13:07:58+00:00 | +1,322.6s | TEST] SCRFD bbox 100/2000; status=VALID
[2026-09-29T13:08:04+00:00 | 

/tmp/ipykernel_58/2815326475.py:564: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


[2026-09-29T13:10:16+00:00 | +1,461.2s | TEST] vanilla_seed100: APCER=0.296375 BPCER=0.023609 ACER=0.159992 AUC=0.957275
[2026-09-29T13:10:16+00:00 | +1,461.3s | TEST] Evaluating csmr_seed100 on the SAME fixed Test-2000 manifest with its locked Val threshold


/tmp/ipykernel_58/2815326475.py:564: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


[2026-09-29T13:10:27+00:00 | +1,472.2s | TEST] csmr_seed100: APCER=0.328358 BPCER=0.016863 ACER=0.172611 AUC=0.958933
POST-FREEZE HELD-OUT IN-DOMAIN CONFIRMATION
| Metric | Vanilla seed100 | CSMR seed100 | Delta (CSMR - Vanilla) |
|---|---:|---:|---:|
| Test APCER | 0.296375 | 0.328358 | +0.031983 |
| Test BPCER | 0.023609 | 0.016863 | -0.006745 |
| Test ACER | 0.159992 | 0.172611 | +0.012619 |
| Test AUC | 0.957275 | 0.958933 | +0.001659 |
[2026-09-29T13:10:27+00:00 | +1,472.2s | TEST] Saved Test-2000 manifest, both predictions, metrics and summary; freeze decision unchanged


In [34]:
# Test results can describe the outcome; they cannot alter the persisted freeze gate.
if strict_json_load(pretest_path)['freeze_gate']['result']!=FREEZE_CHECK:
    raise RuntimeError('Pretest freeze decision changed')
final={'schema':'csmr_v1_freeze_check_seed100_final_v1','training_seed':100,
    'frozen_development_manifest':{'identity':'micro_manifest_v1_seed42.npz',
        'sha256':resources['fingerprints']['micro_manifest_sha256']},
    'resource_fingerprints':resources['fingerprints'],
    'freeze_gate':{'primary_rule':pretest['gate_definitions']['primary_rule'],
        'secondary_rule':pretest['gate_definitions']['secondary_rule'],
        'result':FREEZE_CHECK,'triggered_gate':triggered_gate},
    'vanilla':{k:vanilla[k] for k in ('selected_epoch','locked_threshold','checkpoint_sha256','val','lcc')},
    'csmr':{k:csmr[k] for k in ('selected_epoch','locked_threshold','checkpoint_sha256','val','lcc')},
    'paired_delta':paired_delta,'celeba_test2000_opened':celeba_test_opened,
    'celeba_test2000':celeba_test2000,
    'pretest_summary_sha256':sha256_file(pretest_path)}
final_path=comparison_dir/'final_summary.json'
strict_json_write(final_path,final)
report=['# CSMR-v1 freeze check, training seed 100','',
    '## Purpose',
    'Test whether fixed CSMR-v1 reproduces its cross-domain development gain against a paired Vanilla baseline, then describe in-domain held-out Test-2000 only after a passing freeze gate.','',
    '## Frozen protocol',
    f'Frozen Stage 0 seed42 manifest SHA `{resources["fingerprints"]["micro_manifest_sha256"]}`; 5,000 Train/2,000 Val; training seed 100. Both models independently start from the same ImageNet MobileNetV3-Small policy and use the same crop, augmentation, optimizer, budget and Val-only checkpoint/threshold rules. LCC is external development. Stage 1 PAD weights were not loaded.','',
    '## Seed-100 Vanilla result',
    f'Best epoch {vanilla["selected_epoch"]}; Val ACER {vanilla["val"]["acer"]:.6f}, LCC HTER {vanilla["lcc"]["hter"]:.6f}, LCC AUC {vanilla["lcc"]["auc"]:.6f}.','',
    '## Seed-100 CSMR result',
    f'Best epoch {csmr["selected_epoch"]}; Val ACER {csmr["val"]["acer"]:.6f}, LCC HTER {csmr["lcc"]["hter"]:.6f}, LCC AUC {csmr["lcc"]["auc"]:.6f}.','',
    '## Paired Val/LCC table',paired_table(vanilla,csmr,'development'),'',
    '## Freeze gate',f'**{FREEZE_CHECK}** via `{triggered_gate}`. The decision was written to the pretest JSON/Markdown before Test metadata was opened. This fixed development heuristic is not a statistical-significance test.','',
    '## CSMR diagnostics']
if csmr['diagnostics']:
    d=csmr['diagnostics'][-1]
    report.append(f'Final epoch: LOW/MID/HIGH fractions '+
        '/'.join(f'{d[f"worst_{b.lower()}_fraction"]:.3f}' for b in BANDS)+
        f'; harmful fraction {d["harmful_fraction"]:.3f}; mean harmful margin drop {d["mean_harmful_margin_drop"]}; mean gain {d["mean_sampled_gain"]:.3f}; mean clip fraction {d["mean_clip_fraction"]:.4f}.')
else:
    report.append('No CSMR diagnostic epoch was completed.')
report+=['','## Held-out Test-2000 methodology']
if celeba_test_opened and celeba_test2000 is not None:
    report += [f'Official `metas/intra_test/test_label.json`; deterministic 2,000-sample three-class stratified subset with seed {TEST_SUBSET_SEED}. Ordered manifest SHA `{celeba_test2000["ordered_path_fingerprint_sha256"]}`. Full 2,000/2,000 coverage. Each frozen model used its own micro-Val threshold; Test was not used for checkpoint, threshold or gate selection.',
        f'Test-only E1 v5.3 bbox cache SHA `{celeba_test2000["test_bbox_cache_sha256"]}`; detector SHA `{celeba_test2000["test_bbox_generator_provenance"]["detector_sha256"]}`; status counts `{celeba_test2000["test_bbox_cache_status_counts"]}`. Both models used this exact cache.','',
        '## POST-FREEZE HELD-OUT IN-DOMAIN CONFIRMATION',paired_table(vanilla,csmr,'test'),'']
    if csmr['test']['acer']<=vanilla['test']['acer']+.01 and csmr['test']['auc']>=vanilla['test']['auc']-.01:
        interpretation='The held-out in-domain result is consistent with CSMR improving cross-domain robustness without a major loss of source-domain discrimination.'
    else:
        interpretation='The held-out Test-2000 result shows an in-domain trade-off relative to Vanilla; inspect the paired metrics directly.'
else:
    report += ['Held-out Test-2000 was not opened. No Test metrics are reported.','','## Test-2000 result table','Not applicable.','']
    interpretation='The freeze check did not pass; no held-out Test evidence is available.'
report += ['## Interpretation',interpretation,'',
    '## Limitations','One paired seed, fixed 2,000-sample Test subset, and external-development LCC. The freeze gate is a screening rule, not a significance test. Test remains opened for CSMR-v1 once the marker is written.','',
    '## Recommended next step',
    ('Archive the Test manifest and run independent-seed/larger confirmation without changing the CSMR-v1 settings.' if FREEZE_CHECK=='PASS' else 'Do not scale CSMR-v1 from this failed freeze check; preserve the untouched held-out Test.'),'']
report_path=comparison_dir/'final_report.md'
report_path.write_text('\n'.join(report))
progress('FINAL',f'Final summary={final_path}; report={report_path}; Test opened={celeba_test_opened}')


[2026-09-29T13:10:49+00:00 | +1,493.6s | FINAL] Final summary=/kaggle/working/csmr_v1_freeze_check_seed100/comparison/final_summary.json; report=/kaggle/working/csmr_v1_freeze_check_seed100/comparison/final_report.md; Test opened=True


In [35]:
# LAST CELL: package only this run's output tree, never Kaggle inputs/raw images.
zip_path=Path('/kaggle/working/csmr_v1_freeze_check_seed100.zip')
if zip_path.exists():
    raise RuntimeError(f'Unsafe ZIP collision: {zip_path}; use a clean Kaggle session')
write_zip(output_root,zip_path)
progress('ZIP',f'{zip_path} | {zip_path.stat().st_size/2**20:.2f} MB')
display(FileLink(str(zip_path)))


[2026-09-29T13:10:54+00:00 | +1,499.2s | ZIP] /kaggle/working/csmr_v1_freeze_check_seed100.zip | 20.04 MB


/kaggle/working/csmr_v1_freeze_check_seed100.zip